# Midpoint amplitude and round-trip check -- candidate source for an unverified prose number

Produces `interpolation_systematic/*`. Only a CANDIDATE source for the supplement's "round-trip .12-.16 against a .14 floor" prose (geodesic round-trip 0.124+/-0.020 printed here) -- docs/INVENTORY.md marks this claim unverified, and the `RoundTrip*` macros it might back are defined but never used in the tex.

**Needs:** DTD, the CNT checkpoint.

**Maintained runnable path:** none. This notebook is the best available (unconfirmed) source for that prose number.

In [ ]:
# ============================================================
# Cell 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
results_drive = os.path.join(PROJECT_DIR, "cnt_synthesis")
os.makedirs(results_drive, exist_ok=True)

In [ ]:
# ============================================================
# Cell 2: Install packages
# ============================================================
!pip install -q pot gdown hydra-core omegaconf einops scikit-learn
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# ============================================================
# Cell 3: Clone CNT + fix symlink
# ============================================================
%cd /content
!rm -rf cnt-siga24
!git clone https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24
!pip install -q -r cnt-siga24/requirements.in

ckpts_drive = os.path.join(PROJECT_DIR, "cnt_ckpts")
!rm -rf /content/cnt-siga24/ckpts
!ln -sfn "{ckpts_drive}" /content/cnt-siga24/ckpts
!ls /content/cnt-siga24/ckpts/
%cd /content/cnt-siga24

In [ ]:
# ============================================================
# Cell 4: Load model + imports
# ============================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import copy
import random
import sys

sys.path.insert(0, "/content/cnt-siga24")

import torch.nn.functional as F
from torch import nn
from scipy.optimize import linear_sum_assignment
import ot as pot
from sklearn.decomposition import PCA

from src.inference.core import TexAutoEncoderInference, load_image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = TexAutoEncoderInference(
    ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"),
    device=device,
)
print("CNT model loaded!")

In [ ]:
# ============================================================
# Cell 5: Helper functions
# ============================================================
IMG_SIZE = 256

def load_texture(path):
    return load_image(Path(path), IMG_SIZE).to(device)

def get_dtd_pair(cat_a, cat_b, seed=42):
    random.seed(seed)
    img_a = random.choice(os.listdir(os.path.join(dtd_root, cat_a)))
    img_b = random.choice(os.listdir(os.path.join(dtd_root, cat_b)))
    path_a = os.path.join(dtd_root, cat_a, img_a)
    path_b = os.path.join(dtd_root, cat_b, img_b)
    return load_texture(path_a), load_texture(path_b), path_a, path_b

def encode_texture(img_tensor):
    with torch.no_grad():
        blob = model.encode(img_tensor)
        tree_level = blob["layer_tree_levels"][-1][0]
    n_valid = (tree_level["exist_prob"].squeeze(-1) >= 0.5).sum().item()
    print(f"  Valid textons: {n_valid} / {tree_level['xy'].shape[1]}")
    return blob, tree_level

def get_generator_inputs(img_tensor):
    with torch.no_grad():
        blob = model.encode(img_tensor)
        layout = model.splat(blob)
        bf = layout["layer_level_layouts"][-1][0]["blob_features"].clone()
        si = layout["layer_level_layouts"][-1][0]["score_img"].clone()
    return blob, bf, si

def decode_from_blob(blob):
    with torch.no_grad():
        layout = model.splat(blob)
        return model.decode(layout)

def decode_interp(blob_ref, bf_new, si_new):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf_new
        layout["layer_level_layouts"][-1][0]["score_img"] = si_new
        return model.decode(layout)

def build_style_maps(bf, si, feature_map_sizes):
    scores_pyramid = {}
    s = si
    scores_pyramid[s.shape[-1]] = s
    while s.shape[-1] > min(feature_map_sizes):
        s = F.interpolate(s, s.shape[-1] // 2, mode="bilinear", align_corners=False)
        scores_pyramid[s.shape[-1]] = s
    style_maps = {}
    for size in set(feature_map_sizes):
        style_maps[size] = torch.einsum("bnc,bnhw->bchw", bf, scores_pyramid[size])
    return style_maps

def decode_with_style_maps(blob_ref, style_maps_dict):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        si = layout["layer_level_layouts"][-1][0]["score_img"]
        img_size = si.shape[-1]
        ratios = model.generator.feature_map_size_ratios
        feature_map_sizes = [round(img_size * r) for r in ratios]
        scores_pyramid = {}
        s = si
        scores_pyramid[s.shape[-1]] = s
        while s.shape[-1] > min(feature_map_sizes):
            s = F.interpolate(s, s.shape[-1] // 2, mode="bilinear", align_corners=False)
            scores_pyramid[s.shape[-1]] = s
        feature_in = (
            scores_pyramid[feature_map_sizes[0]][:, 1:]
            .amax(dim=1)[:, None]
            .repeat(1, model.generator.n_embedding, 1, 1)
        )
        x = feature_in.clone()
        for i in range(len(feature_map_sizes)):
            style_map = style_maps_dict[feature_map_sizes[i]]
            x = model.generator.spade_blocks[i](x, style_map)
            if i == len(feature_map_sizes) - 1:
                break
            elif feature_map_sizes[i] != feature_map_sizes[i + 1]:
                x = F.interpolate(x, size=x.shape[-1] * 2, mode="bilinear", align_corners=False)
        recon = torch.nn.functional.tanh(x.unsqueeze(1))
        return recon.flatten(1, 2)

def get_feature_map_sizes():
    ratios = model.generator.feature_map_size_ratios
    return [round(IMG_SIZE * r) for r in ratios]

FMS = get_feature_map_sizes()
print(f"Generator feature map sizes: {FMS}")

def tensor_to_image(tensor):
    img = tensor.squeeze(0).cpu().clamp(-1, 1)
    img = (img + 1) / 2
    return img.permute(1, 2, 0).numpy()

In [ ]:
# ============================================================
# Cell 6: Load texture pairs
# ============================================================
texture_pairs = [
    ("striped", "zigzagged", 42),
    ("dotted", "honeycombed", 42),
    ("woven", "knitted", 42),
    ("cracked", "veined", 42),
    ("braided", "spiralled", 42),
    ("scaly", "cobwebbed", 42),
]

pair_data = {}
for cat_a, cat_b, seed in texture_pairs:
    pair_name = f"{cat_a}_vs_{cat_b}"
    print(f"\nLoading {pair_name}...")
    img_a, img_b, pa, pb = get_dtd_pair(cat_a, cat_b, seed=seed)
    blob_a, tree_a = encode_texture(img_a)
    blob_b, tree_b = encode_texture(img_b)
    blob_a_gen, bf_a, si_a = get_generator_inputs(img_a)
    blob_b_gen, bf_b, si_b = get_generator_inputs(img_b)

    pair_data[pair_name] = {
        "img_a": img_a, "img_b": img_b,
        "blob_a": blob_a, "blob_b": blob_b,
        "tree_a": tree_a, "tree_b": tree_b,
        "blob_a_gen": blob_a_gen, "blob_b_gen": blob_b_gen,
        "bf_a": bf_a, "bf_b": bf_b,
        "si_a": si_a, "si_b": si_b,
        "path_a": pa, "path_b": pb,
    }

print(f"\nLoaded {len(pair_data)} pairs!")

In [ ]:
# ============================================================
# Cell 7: Configure experiment + output structure
# ============================================================

import json
import time
from collections import defaultdict
from itertools import combinations
import torchvision.models as models

# Output directory on Drive
EXPERIMENT_DIR = os.path.join(PROJECT_DIR, "interpolation_systematic")
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
for sub in ["images", "metrics", "nn_analysis", "summaries"]:
    os.makedirs(os.path.join(EXPERIMENT_DIR, sub), exist_ok=True)

print(f"Experiment output: {EXPERIMENT_DIR}")

# DTD texture categories with structural annotations
TEXTURE_TYPES = {
    "periodic": ["chequered", "grid", "honeycombed", "meshed", "polka-dotted",
                 "striped", "zigzagged", "lined"],
    "regular": ["woven", "knitted", "braided", "interlaced", "pleated",
                "perforated", "lacelike"],
    "non_periodic": ["cracked", "veined", "marbled", "fractured", "crystalline"],
    "no_structure": ["bumpy", "matted", "flecked", "speckled", "blotchy",
                     "smeared", "porous"],
    "strong_local": ["dotted", "scaly", "cobwebbed", "spiralled", "swirly",
                     "paisley", "pitted"],
    "fibrous": ["fibrous", "gauzy", "grooved", "stained", "stratified",
                "wrinkled", "sprinkled"],
}

# Filter to categories that actually exist in DTD
available_categories = sorted([c for c in os.listdir(dtd_root)
                                if os.path.isdir(os.path.join(dtd_root, c))])
print(f"Available DTD categories: {len(available_categories)}")

# Map each category to its type(s)
cat_to_type = {}
for ttype, cats in TEXTURE_TYPES.items():
    for c in cats:
        if c in available_categories:
            cat_to_type[c] = ttype

# Categories without a type get "other"
for c in available_categories:
    if c not in cat_to_type:
        cat_to_type[c] = "other"

print(f"Categorized: {len(cat_to_type)} categories")
for ttype in set(cat_to_type.values()):
    cats = [c for c, t in cat_to_type.items() if t == ttype]
    print(f"  {ttype}: {len(cats)} — {', '.join(cats[:5])}{'...' if len(cats) > 5 else ''}")

In [ ]:
# ============================================================
# Cell 9: Sinkhorn OT interpolation implementation
# ============================================================

def sinkhorn_transport(source_feats, target_feats, eps=0.05, n_iters=100):
    """Compute Sinkhorn OT plan between two sets of feature vectors.
    source_feats: (N, D) target_feats: (M, D)
    Returns: transport targets for each source point (N, D)
    """
    N = source_feats.shape[0]
    M = target_feats.shape[0]

    # Cost matrix
    C = torch.cdist(source_feats, target_feats).pow(2)

    # Sinkhorn iterations
    K = torch.exp(-C / eps)
    a = torch.ones(N, 1, device=source_feats.device) / N
    b = torch.ones(M, 1, device=source_feats.device) / M

    u = torch.ones(N, 1, device=source_feats.device)
    for _ in range(n_iters):
        v = b / (K.T @ u + 1e-10)
        u = a / (K @ v + 1e-10)

    # Transport plan
    plan = u * K * v.T  # (N, M)
    # Transport target for each source: weighted average of targets
    plan_normalized = plan / (plan.sum(dim=1, keepdim=True) + 1e-10)
    targets = plan_normalized @ target_feats
    return targets


def interpolate_sinkhorn_pixel(d, eta, n_subsample=4096):
    """Pixel-level Sinkhorn OT interpolation at parameter eta."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    largest = max(FMS)

    source = sm_a[largest][0]  # (C, H, W)
    target = sm_b[largest][0]
    C_ch, H, W = source.shape

    # Flatten to (H*W, C)
    src_flat = source.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = target.permute(1, 2, 0).reshape(-1, C_ch)

    # Subsample
    N = src_flat.shape[0]
    idx_src = torch.randperm(N, device=device)[:n_subsample]
    idx_tgt = torch.randperm(N, device=device)[:n_subsample]

    src_sub = src_flat[idx_src]
    tgt_sub = tgt_flat[idx_tgt]

    # Compute transport targets
    with torch.no_grad():
        transport_targets = sinkhorn_transport(src_sub, tgt_sub, eps=0.05, n_iters=100)

    # Extend to all pixels via nearest-neighbor
    dists_to_sub = torch.cdist(src_flat, src_sub)
    nn_idx = dists_to_sub.argmin(dim=1)
    all_targets = transport_targets[nn_idx]

    # Displacement interpolation
    interp_flat = (1 - eta) * src_flat + eta * all_targets
    interp_sm = interp_flat.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)

    # Build full style_maps dict
    style_maps = {}
    for size in set(FMS):
        if size == largest:
            style_maps[size] = interp_sm
        else:
            style_maps[size] = F.interpolate(interp_sm, size=size,
                                              mode="bilinear", align_corners=False)

    return decode_with_style_maps(d["blob_a_gen"], style_maps)


def interpolate_linear_pixel(d, eta):
    """Pixel-level linear interpolation at parameter eta."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)

    style_maps = {}
    for size in set(FMS):
        style_maps[size] = (1 - eta) * sm_a[size] + eta * sm_b[size]

    return decode_with_style_maps(d["blob_a_gen"], style_maps)


def interpolate_linear_texton(d, eta):
    """Texton-level linear interpolation at parameter eta."""
    bf = (1 - eta) * d["bf_a"] + eta * d["bf_b"]
    si = (1 - eta) * d["si_a"] + eta * d["si_b"]
    return decode_interp(d["blob_a_gen"], bf, si)


print("Interpolation methods ready")

In [ ]:
# ============================================================
# Cell 10: Build ALL texture pairs to test
# ============================================================

print("=" * 70)
print("TASK 1: Building texture pairs")
print("=" * 70)

# Strategy: pick pairs that are
# (a) within the same type (periodic↔periodic, etc.)
# (b) across different types (periodic↔non_periodic, etc.)
# For each category, pick 1 representative image

random.seed(42)
category_images = {}
category_data = {}

for cat in available_categories:
    cat_dir = os.path.join(dtd_root, cat)
    imgs = sorted([f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))])
    if len(imgs) == 0:
        continue
    img_name = random.choice(imgs)
    img_path = os.path.join(cat_dir, img_name)

    try:
        img_tensor = load_texture(img_path)
        blob_gen, bf, si = get_generator_inputs(img_tensor)
        category_images[cat] = img_path
        category_data[cat] = {
            "blob_a_gen": blob_gen, "bf_a": bf, "si_a": si,
            "path_a": img_path,
        }
    except Exception as e:
        print(f"  Skipped {cat}: {e}")

print(f"Loaded {len(category_data)} categories")

# Build pairs: all within-type and a selection of cross-type
all_pairs = []

# Within-type pairs
for ttype, cats in TEXTURE_TYPES.items():
    valid = [c for c in cats if c in category_data]
    for c1, c2 in combinations(valid, 2):
        all_pairs.append((c1, c2, "within", ttype))

# Cross-type pairs (sample to keep manageable)
type_names = list(TEXTURE_TYPES.keys())
for t1, t2 in combinations(type_names, 2):
    valid_t1 = [c for c in TEXTURE_TYPES.get(t1, []) if c in category_data]
    valid_t2 = [c for c in TEXTURE_TYPES.get(t2, []) if c in category_data]
    if not valid_t1 or not valid_t2:
        continue
    # Pick up to 3 cross-type pairs per combination
    random.shuffle(valid_t1)
    random.shuffle(valid_t2)
    for i in range(min(3, len(valid_t1))):
        for j in range(min(1, len(valid_t2))):
            all_pairs.append((valid_t1[i], valid_t2[j], "cross", f"{t1}↔{t2}"))

print(f"Total pairs to test: {len(all_pairs)}")
print(f"  Within-type: {sum(1 for p in all_pairs if p[2] == 'within')}")
print(f"  Cross-type:  {sum(1 for p in all_pairs if p[2] == 'cross')}")


In [ ]:
# ============================================================
# Cell P1: Prop-2 check — measurement functions
# ============================================================
from scipy.optimize import linear_sum_assignment

def centered_amplitude(F):
    """Mean centered feature norm = 'contrast' of the feature cloud (N, D)."""
    return (F - F.mean(0, keepdim=True)).norm(dim=1).mean().item()

def exact_ot_targets(X, Y):
    """Exact OT (Hungarian) — the un-regularized reference. X, Y: (n, D)."""
    C = torch.cdist(X, Y).pow(2).cpu().numpy()
    r, c = linear_sum_assignment(C)
    T = torch.empty_like(X)
    T[r] = Y[c]
    return T

def log_sinkhorn_transport(X, Y, eps=0.05, n_iters=200):
    """Log-domain Sinkhorn — numerically stable at small eps.
    (Kernel-domain version underflows below eps ~ 0.02: exp(-C/eps) -> 0.)"""
    C = torch.cdist(X, Y).pow(2)
    n, m = C.shape
    log_a = torch.full((n,), -np.log(n), device=X.device)
    log_b = torch.full((m,), -np.log(m), device=X.device)
    f = torch.zeros(n, device=X.device)
    g = torch.zeros(m, device=X.device)
    for _ in range(n_iters):
        g = eps * (log_b - torch.logsumexp((f[:, None] - C) / eps, dim=0))
        f = eps * (log_a - torch.logsumexp((g[None, :] - C) / eps, dim=1))
    log_plan = (f[:, None] + g[None, :] - C) / eps
    plan = torch.exp(log_plan)
    plan = plan / (plan.sum(dim=1, keepdim=True) + 1e-30)
    return plan @ Y

def prop2_measure_pair(d, n_sub=4096, n_exact=1500, eps_list=(0.05, 0.01), n_t=11):
    """Amplitude trajectory under spatial coupling (= your linear blend),
    your Sinkhorn, log-domain Sinkhorn, and exact OT — on largest style map."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    largest = max(FMS)
    X_full = sm_a[largest][0].permute(1, 2, 0).reshape(-1, sm_a[largest].shape[1])
    Y_full = sm_b[largest][0].permute(1, 2, 0).reshape(-1, sm_b[largest].shape[1])

    # SAME indices for X and Y: preserves the spatial coupling that
    # interpolate_linear_pixel actually uses. Do NOT permute independently.
    idx = torch.randperm(X_full.shape[0], device=X_full.device)[:n_sub]
    X, Y = X_full[idx], Y_full[idx]

    # empirical alignment rho under the spatial coupling
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    rho = ((Xc * Yc).sum(1).mean() /
           (Xc.norm(dim=1).mean() * Yc.norm(dim=1).mean() + 1e-12)).item()

    # transport targets
    with torch.no_grad():
        T_sink = {e: sinkhorn_transport(X, Y, eps=e, n_iters=100) for e in eps_list}
        T_log  = {e: log_sinkhorn_transport(X, Y, eps=e) for e in eps_list}
    Xe, Ye = X[:n_exact], Y[:n_exact]
    T_ex = exact_ot_targets(Xe, Ye)

    ts = np.linspace(0, 1, n_t)
    def traj(A, B): return [centered_amplitude((1 - t) * A + t * B) for t in ts]
    a_lin = traj(X, Y)
    a0, a1 = a_lin[0], a_lin[-1]
    chord = [(1 - t) * a0 + t * a1 for t in ts]
    mid = n_t // 2

    res = {
        "rho": rho,
        "pred_factor": float(np.sqrt((1 + rho) / 2)),   # theory: lin mid/chord
        "lin_mid_over_chord": a_lin[mid] / chord[mid],
        "exact_mid_over_chord": traj(Xe, T_ex)[mid] / chord[mid],
        "lin_dips_below_both": min(a_lin) < min(a0, a1) - 1e-6,
        "t_star_pred": a0**2 / (a0**2 + a1**2),
        "t_star_obs": float(ts[int(np.argmin(a_lin))]),
        "amp_lin": a_lin, "chord": chord, "ts": ts.tolist(),
        "endpoints": (a0, a1),
    }
    for e in eps_list:
        res[f"sink_e{e}_mid_over_chord"] = traj(X, T_sink[e])[mid] / chord[mid]
        res[f"logsink_e{e}_mid_over_chord"] = traj(X, T_log[e])[mid] / chord[mid]
    return res

print("Prop-2 measurement functions ready")

In [ ]:
# ============================================================
# Cell P2: Run Prop-2 check over pairs
# ============================================================
import json as _json

P2_MAX_PAIRS = 30          # set None for all pairs
p2_results = []
p2_pairs = all_pairs if P2_MAX_PAIRS is None else all_pairs[:P2_MAX_PAIRS]

for k, (cat_a, cat_b, pair_type, type_label) in enumerate(p2_pairs):
    if cat_a not in category_data or cat_b not in category_data:
        continue
    d = {
        "bf_a": category_data[cat_a]["bf_a"], "si_a": category_data[cat_a]["si_a"],
        "bf_b": category_data[cat_b]["bf_a"], "si_b": category_data[cat_b]["si_a"],
    }
    try:
        r = prop2_measure_pair(d)
    except Exception as ex:
        print(f"  skip {cat_a} vs {cat_b}: {ex}")
        continue
    r.update(pair=f"{cat_a}_vs_{cat_b}", pair_type=pair_type, type_label=type_label)
    p2_results.append(r)
    print(f"[{k+1}/{len(p2_pairs)}] {r['pair'][:34]:34s} "
          f"rho={r['rho']:+.3f}  lin={r['lin_mid_over_chord']:.3f} "
          f"(pred {r['pred_factor']:.3f})  exact={r['exact_mid_over_chord']:.3f}  "
          f"sink.05={r['sink_e0.05_mid_over_chord']:.3f}  "
          f"t*={r['t_star_pred']:.2f}/{r['t_star_obs']:.2f}")

with open(os.path.join(OUTPUT_ROOT, "prop2_results.json"), "w") as f:
    _json.dump(p2_results, f, indent=1)
print(f"\nSaved {len(p2_results)} pairs -> prop2_results.json")

In [ ]:
OUTPUT_ROOT = "."   # or whatever your Cell 7 output-dir variable is called
with open(os.path.join(OUTPUT_ROOT, "prop2_results.json"), "w") as f:
    _json.dump(p2_results, f, indent=1)
print(f"Saved {len(p2_results)} pairs")

In [ ]:
# ============================================================
# Cell P3: Prop-2 summary + figures
# ============================================================
lin_obs  = np.array([r["lin_mid_over_chord"] for r in p2_results])
lin_pred = np.array([r["pred_factor"] for r in p2_results])
exact    = np.array([r["exact_mid_over_chord"] for r in p2_results])
sink05   = np.array([r["sink_e0.05_mid_over_chord"] for r in p2_results])
rhos     = np.array([r["rho"] for r in p2_results])
dips     = np.array([r["lin_dips_below_both"] for r in p2_results])
tstar_er = np.array([abs(r["t_star_pred"] - r["t_star_obs"]) for r in p2_results])

print("=" * 68)
print("PROP-2 CHECK SUMMARY")
print("=" * 68)
print(f"pairs measured:                    {len(p2_results)}")
print(f"mean rho (spatial coupling):       {rhos.mean():+.3f}  (theory expects ~0)")
print(f"linear mid/chord:                  {lin_obs.mean():.3f} ± {lin_obs.std():.3f}")
print(f"predicted sqrt((1+rho)/2):         {lin_pred.mean():.3f} ± {lin_pred.std():.3f}")
print(f"mean |observed - predicted|:       {np.abs(lin_obs - lin_pred).mean():.4f}")
print(f"exact OT mid/chord:                {exact.mean():.3f} ± {exact.std():.3f}  (theory: ~1)")
print(f"your Sinkhorn e=0.05 mid/chord:    {sink05.mean():.3f} ± {sink05.std():.3f}")
print(f"  -> entropic shrinkage vs exact:  {(exact - sink05).mean():+.3f}")
print(f"linear dips below BOTH endpoints:  {dips.sum()}/{len(dips)} pairs")
print(f"t* prediction error (mean):        {tstar_er.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.scatter(lin_pred, lin_obs, s=28, alpha=0.7, label="linear blend")
lims = [min(lin_pred.min(), lin_obs.min()) - 0.02, 1.02]
ax.plot(lims, lims, "k--", lw=1, label="theory = observation")
ax.axvline(1/np.sqrt(2), color="gray", ls=":", lw=1, label=r"$1/\sqrt{2}$")
ax.set_xlabel(r"predicted $\sqrt{(1+\rho)/2}$"); ax.set_ylabel("observed mid/chord")
ax.set_title("Prop 2: collapse prediction vs observation"); ax.legend(fontsize=8)

ax = axes[1]
ts = p2_results[0]["ts"]
lin_curves = np.array([np.array(r["amp_lin"]) / np.array(r["chord"]) for r in p2_results])
ax.plot(ts, lin_curves.mean(0), "o-", color="#c0392b", label="linear (mean over pairs)")
ax.fill_between(ts, lin_curves.mean(0) - lin_curves.std(0),
                lin_curves.mean(0) + lin_curves.std(0), color="#c0392b", alpha=0.15)
ax.axhline(1.0, color="k", ls="--", lw=1, label="geodesic chord")
ax.axhline(1/np.sqrt(2), color="gray", ls=":", lw=1, label=r"$1/\sqrt{2}$")
ax.set_xlabel("t"); ax.set_ylabel("amplitude / chord")
ax.set_title("Amplitude trajectory (style-map features)"); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "prop2_summary.png"), dpi=150)
plt.show()

In [ ]:
for r in p2_results:
    print(f"{r['pair'][:34]:34s} logsink e=.05: {r['logsink_e0.05_mid_over_chord']:.3f}   "
          f"e=.01: {r['logsink_e0.01_mid_over_chord']:.3f}   exact: {r['exact_mid_over_chord']:.3f}")

In [ ]:
# ============================================================
# Cell 8: VGG perceptual distance + metrics functions
# ============================================================

class VGGPerceptualDistance:
    """Extract VGG16 features at conv1_2, conv2_2, conv3_3, conv4_3
    and compute L2 distance between feature maps."""

    def __init__(self, device='cuda'):
        vgg = models.vgg16(pretrained=True).features.eval().to(device)
        for p in vgg.parameters():
            p.requires_grad = False
        self.device = device

        # Hook into the 4 target layers
        self.features = {}
        target_layers = {3: 'conv1_2', 8: 'conv2_2', 15: 'conv3_3', 22: 'conv4_3'}
        self.slices = []
        prev = 0
        for idx in sorted(target_layers.keys()):
            self.slices.append(torch.nn.Sequential(*list(vgg.children())[prev:idx+1]))
            prev = idx + 1
        for s in self.slices:
            s.eval().to(device)

        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    def extract(self, img):
        """img: (1, 3, 256, 256) in [-1, 1]"""
        x = (img + 1) / 2  # to [0, 1]
        x = (x - self.mean) / self.std
        feats = []
        for s in self.slices:
            x = s(x)
            feats.append(x.clone())
        return feats

    def distance(self, img_a, img_b):
        """Compute perceptual distance between two images."""
        feats_a = self.extract(img_a)
        feats_b = self.extract(img_b)
        d = 0
        for fa, fb in zip(feats_a, feats_b):
            d += (fa - fb).pow(2).sum().item()
        return d

    def distance_batch(self, img_a, images):
        """Distance from img_a to a list of images."""
        feats_a = self.extract(img_a)
        dists = []
        for img in images:
            feats_b = self.extract(img)
            d = sum((fa - fb).pow(2).sum().item() for fa, fb in zip(feats_a, feats_b))
            dists.append(d)
        return dists


vgg_dist = VGGPerceptualDistance(device=device)
print("VGG perceptual distance ready")


def compute_interpolation_metrics(distances, n_steps=21):
    """Given normalized distances at each eta, compute metrics."""
    etas = np.linspace(0, 1, n_steps)

    # Path linearity R²
    from scipy.stats import linregress
    slope, intercept, r_value, _, _ = linregress(etas, distances)
    r_squared = r_value ** 2

    # Midpoint ratio
    mid_idx = n_steps // 2
    midpoint_ratio = distances[mid_idx]

    # Monotonicity (CV of step sizes)
    steps = np.diff(distances)
    if np.mean(np.abs(steps)) > 1e-10:
        monotonicity = np.std(steps) / np.mean(np.abs(steps))
    else:
        monotonicity = 0

    # Symmetry (how similar is forward vs backward)
    forward = distances
    backward = 1 - distances[::-1]
    symmetry_error = np.mean(np.abs(forward - backward))

    return {
        "r_squared": float(r_squared),
        "midpoint_ratio": float(midpoint_ratio),
        "monotonicity": float(monotonicity),
        "symmetry_error": float(symmetry_error),
    }

print("Metrics functions ready")


In [ ]:
# ============================================================
# Cell P4: Decode midpoints — linear vs sink e=.05 vs logsink e=.01 vs exact OT
# ============================================================

P4_PAIRS = [("chequered", "grid"),
            ("woven", "knitted"),
            ("polka-dotted", "striped"),
            ("striped", "zigzagged")]      # <- H3-violation case
P4_ETA = 0.5
P4_N_SUB = 4096
P4_N_EXACT = 2000       # Hungarian is O(n^3); 2000 ~ tens of seconds per pair

def decode_with_transport(d, eta, transport_fn, n_subsample=P4_N_SUB):
    """Same flow as interpolate_sinkhorn_pixel, but transport_fn is pluggable."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    largest = max(FMS)
    source, target = sm_a[largest][0], sm_b[largest][0]
    C_ch, H, W = source.shape
    src_flat = source.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = target.permute(1, 2, 0).reshape(-1, C_ch)

    N = src_flat.shape[0]
    idx_src = torch.randperm(N, device=src_flat.device)[:n_subsample]
    idx_tgt = torch.randperm(N, device=src_flat.device)[:n_subsample]
    src_sub, tgt_sub = src_flat[idx_src], tgt_flat[idx_tgt]

    with torch.no_grad():
        transport_targets = transport_fn(src_sub, tgt_sub)

    # extend to all pixels via NN (as in your Cell 9)
    nn_idx = torch.cdist(src_flat, src_sub).argmin(dim=1)
    all_targets = transport_targets[nn_idx]

    interp_flat = (1 - eta) * src_flat + eta * all_targets
    interp_sm = interp_flat.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
    style_maps = {}
    for size in set(FMS):
        style_maps[size] = interp_sm if size == largest else F.interpolate(
            interp_sm, size=size, mode="bilinear", align_corners=False)
    return decode_with_style_maps(d["blob_a_gen"], style_maps)

METHODS = {
    "linear":       lambda d, eta: interpolate_linear_pixel(d, eta),
    "sink e=.05":   lambda d, eta: decode_with_transport(
                        d, eta, lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100)),
    "logsink e=.01": lambda d, eta: decode_with_transport(
                        d, eta, lambda X, Y: log_sinkhorn_transport(X, Y, eps=0.01, n_iters=500)),
    "exact OT":     lambda d, eta: decode_with_transport(
                        d, eta, exact_ot_targets, n_subsample=P4_N_EXACT),
}

have_vgg = "vgg_dist" in globals()
cols = ["A"] + list(METHODS.keys()) + ["B"]
fig, axes = plt.subplots(len(P4_PAIRS), len(cols),
                         figsize=(3.2 * len(cols), 3.4 * len(P4_PAIRS)))
p4_metrics = []

for row, (cat_a, cat_b) in enumerate(P4_PAIRS):
    d = {"blob_a_gen": category_data[cat_a]["blob_a_gen"],
         "bf_a": category_data[cat_a]["bf_a"], "si_a": category_data[cat_a]["si_a"],
         "bf_b": category_data[cat_b]["bf_a"], "si_b": category_data[cat_b]["si_a"]}
    with torch.no_grad():
        img_a = decode_interp(category_data[cat_a]["blob_a_gen"],
                              d["bf_a"], d["si_a"])
        img_b = decode_interp(category_data[cat_b]["blob_a_gen"],
                              d["bf_b"], d["si_b"])
    imgs = {"A": img_a, "B": img_b}
    for name, fn in METHODS.items():
        with torch.no_grad():
            imgs[name] = fn(d, P4_ETA)

    row_metrics = {"pair": f"{cat_a}_vs_{cat_b}"}
    for col, name in enumerate(cols):
        ax = axes[row, col]
        ax.imshow(tensor_to_image(imgs[name])); ax.axis("off")
        title = name
        if have_vgg and name in METHODS:
            dA = vgg_dist.distance(imgs[name], img_a)
            dB = vgg_dist.distance(imgs[name], img_b)
            total = vgg_dist.distance(img_a, img_b)
            # midpoint quality: sum of dists / endpoint dist (1.0 = on a VGG "straight line"),
            # balance: |dA-dB| / total (0 = perfectly centered)
            row_metrics[name] = {"detour": (dA + dB) / total,
                                 "imbalance": abs(dA - dB) / total}
            title += f"\ndetour {(dA+dB)/total:.2f} | bal {abs(dA-dB)/total:.2f}"
        if row == 0 or (have_vgg and name in METHODS):
            ax.set_title(title, fontsize=8)
        if col == 0:
            ax.set_ylabel(f"{cat_a}\nvs {cat_b}", fontsize=9)
    p4_metrics.append(row_metrics)

plt.suptitle(f"Midpoint (η={P4_ETA}) decoded under each transport", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p4_midpoint_decode_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

if have_vgg:
    print(f"\n{'pair':28s}" + "".join(f"{m:>18s}" for m in METHODS))
    for rm in p4_metrics:
        print(f"{rm['pair']:28s}" + "".join(
            f"  det {rm[m]['detour']:.2f}/bal {rm[m]['imbalance']:.2f}" for m in METHODS))
else:
    print("vgg_dist not defined — visual comparison only. "
          "Copy Cell 8 from the main notebook for VGG metrics.")

In [ ]:
# ============================================================
# Cell P5: Common-mode test — does mean-subtraction explain rho≈0.44?
# Hypothesis: style maps = shared positional component c(x) + texture residual s(x),
# with rho(s)≈0. If true: residual rho drops toward 0 and sqrt((1+rho)/2) fits again.
# ============================================================

# build the common mode: mean style map across ALL categories, per position
largest = max(FMS)
with torch.no_grad():
    all_maps = torch.stack([
        build_style_maps(category_data[c]["bf_a"], category_data[c]["si_a"], FMS)[largest][0]
        for c in category_data
    ])                                   # (n_cat, C, H, W)
common_mode = all_maps.mean(0)           # (C, H, W)
cm_flat = common_mode.permute(1, 2, 0).reshape(-1, common_mode.shape[0])

feat_norm = all_maps.permute(0, 2, 3, 1).reshape(-1, all_maps.shape[1]).norm(dim=1).mean()
print(f"common-mode norm / mean feature norm: "
      f"{cm_flat.norm(dim=1).mean().item() / feat_norm.item():.3f}  (large => shared component is real)")

def rho_and_collapse(X, Y, n_t=11):
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    rho = ((Xc * Yc).sum(1).mean() /
           (Xc.norm(dim=1).mean() * Yc.norm(dim=1).mean() + 1e-12)).item()
    ts = np.linspace(0, 1, n_t)
    a = [centered_amplitude((1 - t) * X + t * Y) for t in ts]
    chord = (a[0] + a[-1]) / 2
    return rho, a[n_t // 2] / chord

p5_rows = []
for (cat_a, cat_b, *_ ) in p2_pairs:
    if cat_a not in category_data or cat_b not in category_data:
        continue
    sm_a = build_style_maps(category_data[cat_a]["bf_a"], category_data[cat_a]["si_a"], FMS)[largest][0]
    sm_b = build_style_maps(category_data[cat_b]["bf_a"], category_data[cat_b]["si_a"], FMS)[largest][0]
    Xf = sm_a.permute(1, 2, 0).reshape(-1, sm_a.shape[0])
    Yf = sm_b.permute(1, 2, 0).reshape(-1, sm_b.shape[0])
    idx = torch.randperm(Xf.shape[0], device=Xf.device)[:4096]
    X, Y = Xf[idx], Yf[idx]
    Xr, Yr = X - cm_flat[idx], Y - cm_flat[idx]          # residuals
    rho_raw, col_raw = rho_and_collapse(X, Y)
    rho_res, col_res = rho_and_collapse(Xr, Yr)
    p5_rows.append((f"{cat_a}_vs_{cat_b}", rho_raw, col_raw, rho_res, col_res,
                    float(np.sqrt((1 + rho_res) / 2))))

print(f"\n{'pair':32s}{'rho_raw':>8s}{'col_raw':>8s}{'rho_res':>9s}{'col_res':>9s}{'pred_res':>9s}")
for row in p5_rows:
    print(f"{row[0][:32]:32s}{row[1]:8.3f}{row[2]:8.3f}{row[3]:9.3f}{row[4]:9.3f}{row[5]:9.3f}")

rr = np.array([r[3] for r in p5_rows]); cr = np.array([r[4] for r in p5_rows])
pr = np.array([r[5] for r in p5_rows])
print(f"\nresidual rho mean:          {rr.mean():+.3f}   (was ~+0.44 raw; hypothesis: ~0)")
print(f"residual |obs - pred| mean: {np.abs(cr - pr).mean():.4f}  (hypothesis: <~0.03)")

In [ ]:
# ============================================================
# Cell P6: H1 shuffle test — does spatial arrangement carry identity?
# Block-shuffle the style map (preserves marginal, destroys arrangement), decode,
# measure VGG distance to the original decode. H1 predicts: small for stochastic
# textures, large for regular ones (chequered, zigzagged...).
# ============================================================

P6_BLOCK = 4          # shuffle in bxb blocks of style-map cells (local coherence kept)
P6_CATS = list(category_data.keys())   # or a subset

def block_shuffle_map(sm, b):
    """sm: (1, C, H, W) -> spatially permute bxb blocks."""
    _, C, H, W = sm.shape
    Hc, Wc = (H // b) * b, (W // b) * b
    x = sm[:, :, :Hc, :Wc]
    x = x.reshape(1, C, Hc // b, b, Wc // b, b).permute(0, 2, 4, 1, 3, 5)  # (1,hb,wb,C,b,b)
    nb = (Hc // b) * (Wc // b)
    x = x.reshape(1, nb, C, b, b)[:, torch.randperm(nb, device=sm.device)]
    x = x.reshape(1, Hc // b, Wc // b, C, b, b).permute(0, 3, 1, 4, 2, 5).reshape(1, C, Hc, Wc)
    out = sm.clone(); out[:, :, :Hc, :Wc] = x
    return out

def decode_shuffled(cat, b):
    sm = build_style_maps(category_data[cat]["bf_a"], category_data[cat]["si_a"], FMS)
    shuf = block_shuffle_map(sm[largest], b)
    style_maps = {}
    for size in set(FMS):
        style_maps[size] = shuf if size == largest else F.interpolate(
            shuf, size=size, mode="bilinear", align_corners=False)
    return decode_with_style_maps(category_data[cat]["blob_a_gen"], style_maps)

def decode_original(cat):
    return decode_interp(category_data[cat]["blob_a_gen"],
                         category_data[cat]["bf_a"], category_data[cat]["si_a"])

# scale: typical inter-category VGG distance
ref_cats = P6_CATS[:8]
with torch.no_grad():
    ref_imgs = {c: decode_original(c) for c in ref_cats}
inter = [vgg_dist.distance(ref_imgs[a], ref_imgs[b])
         for i, a in enumerate(ref_cats) for b in ref_cats[i+1:]]
inter_scale = float(np.mean(inter))

p6_rows = []
for cat in P6_CATS:
    with torch.no_grad():
        orig = ref_imgs[cat] if cat in ref_imgs else decode_original(cat)
        shuf = decode_shuffled(cat, P6_BLOCK)
        d = vgg_dist.distance(shuf, orig) / inter_scale
    p6_rows.append((cat, d))
    print(f"{cat:20s} shuffle-shift / inter-texture scale: {d:.3f}")

p6_rows.sort(key=lambda r: r[1])
print(f"\nmost marginal-sufficient (H1 holds):   {[r[0] for r in p6_rows[:3]]}")
print(f"most arrangement-dependent (H1 weak):  {[r[0] for r in p6_rows[-3:]]}")
print("prediction: regular/oriented textures (chequered, zigzagged, striped...) at the bottom")

In [ ]:
# ============================================================
# Cell P7: H3 quantified — feature-cloud bimodality vs exact-OT midpoint drop
# Prediction: pairs whose clouds are more bimodal have lower exact_mid_over_chord.
# Requires p2_results from Cell P2.
# ============================================================
from sklearn.cluster import KMeans
from scipy.stats import spearmanr, pearsonr

def bimodality_score(cat, n=3000):
    sm = build_style_maps(category_data[cat]["bf_a"], category_data[cat]["si_a"], FMS)[largest][0]
    Ffull = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
    F_ = Ffull[torch.randperm(Ffull.shape[0])[:n]].cpu().numpy()
    km = KMeans(n_clusters=2, n_init=5, random_state=0).fit(F_)
    c0, c1 = km.cluster_centers_
    lab = km.labels_
    s0 = np.linalg.norm(F_[lab == 0] - c0, axis=1).mean()
    s1 = np.linalg.norm(F_[lab == 1] - c1, axis=1).mean()
    sep = np.linalg.norm(c0 - c1) / (s0 + s1 + 1e-9)      # mode separation
    balance = min((lab == 0).mean(), (lab == 1).mean())    # 0.5 = balanced modes
    return sep * (2 * balance)                             # separated AND balanced => bimodal

bimod = {}
for cat in category_data:
    bimod[cat] = bimodality_score(cat)
for cat, s in sorted(bimod.items(), key=lambda kv: -kv[1]):
    print(f"{cat:20s} bimodality {s:.3f}")

pair_bm, pair_exact = [], []
for r in p2_results:
    ca, cb = r["pair"].split("_vs_")
    if ca in bimod and cb in bimod:
        pair_bm.append(max(bimod[ca], bimod[cb]))
        pair_exact.append(r["exact_mid_over_chord"])
pair_bm, pair_exact = np.array(pair_bm), np.array(pair_exact)
sp = spearmanr(pair_bm, pair_exact); pe = pearsonr(pair_bm, pair_exact)
print(f"\nbimodality vs exact-OT mid/chord:  spearman r={sp.statistic:+.3f} (p={sp.pvalue:.1e}), "
      f"pearson r={pe.statistic:+.3f}")
print("prediction: significantly NEGATIVE, driven by (but not only) zigzagged pairs")

plt.figure(figsize=(5.5, 4))
plt.scatter(pair_bm, pair_exact, s=25, alpha=0.7)
plt.xlabel("pair bimodality (max of endpoints)"); plt.ylabel("exact OT mid/chord")
plt.title("H3: multimodality predicts geodesic amplitude drop")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell P8: Pixel-level contrast + decode-side symmetry check
# (a) Does the amplitude ordering survive the decoder? contrast(mid)/mean(contrast A,B)
#     prediction: linear < sink.05 <= logsink.01 <= exact
# (b) Is the balance asymmetry decode-side? Decode from B's side too:
#     prediction: A-side hugs A, B-side hugs B (mirrored imbalance) => motivates OT-sym.
# Requires decode_with_transport from Cell P4.
# ============================================================

def img_contrast(t):
    arr = np.asarray(tensor_to_image(t), dtype=float)
    return arr.mean(-1).std()

P8_PAIRS = [("chequered", "grid"), ("woven", "knitted"),
            ("polka-dotted", "striped"), ("striped", "zigzagged")]

print(f"{'pair':26s}{'linear':>8s}{'s.05':>8s}{'ls.01':>8s}{'exact':>8s}   balA(A/B-side)")
p8 = []
for cat_a, cat_b in P8_PAIRS:
    dA = {"blob_a_gen": category_data[cat_a]["blob_a_gen"],
          "bf_a": category_data[cat_a]["bf_a"], "si_a": category_data[cat_a]["si_a"],
          "bf_b": category_data[cat_b]["bf_a"], "si_b": category_data[cat_b]["si_a"]}
    dB = {"blob_a_gen": category_data[cat_b]["blob_a_gen"],          # decode from B side
          "bf_a": category_data[cat_b]["bf_a"], "si_a": category_data[cat_b]["si_a"],
          "bf_b": category_data[cat_a]["bf_a"], "si_b": category_data[cat_a]["si_a"]}
    with torch.no_grad():
        img_a = decode_interp(dA["blob_a_gen"], dA["bf_a"], dA["si_a"])
        img_b = decode_interp(dB["blob_a_gen"], dB["bf_a"], dB["si_a"])
        mids = {
            "linear": interpolate_linear_pixel(dA, 0.5),
            "s.05":   decode_with_transport(dA, 0.5,
                        lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100)),
            "ls.01":  decode_with_transport(dA, 0.5,
                        lambda X, Y: log_sinkhorn_transport(X, Y, eps=0.01, n_iters=500)),
            "exact":  decode_with_transport(dA, 0.5, exact_ot_targets, n_subsample=2000),
        }
        mid_Bside = decode_with_transport(dB, 0.5,
                        lambda X, Y: log_sinkhorn_transport(X, Y, eps=0.01, n_iters=500))
    cA, cB = img_contrast(img_a), img_contrast(img_b)
    ratios = {k: img_contrast(v) / ((cA + cB) / 2) for k, v in mids.items()}
    total = vgg_dist.distance(img_a, img_b)
    balA = (vgg_dist.distance(mids["ls.01"], img_a) - vgg_dist.distance(mids["ls.01"], img_b)) / total
    balB = (vgg_dist.distance(mid_Bside, img_a) - vgg_dist.distance(mid_Bside, img_b)) / total
    p8.append((f"{cat_a}_vs_{cat_b}", ratios, balA, balB))
    print(f"{cat_a+'_vs_'+cat_b:26s}"
          f"{ratios['linear']:8.3f}{ratios['s.05']:8.3f}{ratios['ls.01']:8.3f}{ratios['exact']:8.3f}"
          f"   {balA:+.2f} / {balB:+.2f}")

print("\ncontrast: ordering linear < OT methods confirms Prop 2 reaches pixels")
print("balance: signed; A-side negative (closer to A) and B-side positive => "
      "asymmetry is decode-side, OT-sym is the fix")

In [ ]:
# ============================================================
# Cell P9: Timing accounting — wall-clock per midpoint, per method
# (hygiene for the 215x claim: same hardware, decode included, medians)
# ============================================================
import time

cat_a, cat_b = "woven", "knitted"
d = {"blob_a_gen": category_data[cat_a]["blob_a_gen"],
     "bf_a": category_data[cat_a]["bf_a"], "si_a": category_data[cat_a]["si_a"],
     "bf_b": category_data[cat_b]["bf_a"], "si_b": category_data[cat_b]["si_a"]}

timed = {
    "linear": lambda: interpolate_linear_pixel(d, 0.5),
    "sink e=.05": lambda: decode_with_transport(d, 0.5,
        lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100)),
    "logsink e=.01": lambda: decode_with_transport(d, 0.5,
        lambda X, Y: log_sinkhorn_transport(X, Y, eps=0.01, n_iters=500)),
    "exact OT (n=2000)": lambda: decode_with_transport(d, 0.5, exact_ot_targets,
                                                       n_subsample=2000),
}
for name, fn in timed.items():
    with torch.no_grad():
        fn()                                   # warmup
        times = []
        for _ in range(3):
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t0 = time.time(); fn()
            if torch.cuda.is_available(): torch.cuda.synchronize()
            times.append(time.time() - t0)
    print(f"{name:22s} median {np.median(times)*1000:8.1f} ms per midpoint (incl. decode)")

In [ ]:
# ============================================================
# Cell P10 (self-contained): Link 2 — does shuffle-shift (H1 score)
# predict OT's win margin? Computes margins directly, no saved JSON needed.
# Margin = VGG detour(linear) - detour(sink e=.05); positive = OT better.
# Prediction: NEGATIVE correlation with shuffle-shift.
# Requires: p2_pairs (P2), p6_rows (P6), decode_with_transport (P4), vgg_dist.
# ============================================================
from scipy.stats import spearmanr, pearsonr

shuffle_score = dict(p6_rows)

def midpoint_detour(d, img_a, img_b, total, method):
    with torch.no_grad():
        if method == "linear":
            mid = interpolate_linear_pixel(d, 0.5)
        else:
            mid = decode_with_transport(d, 0.5,
                lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100))
        return (vgg_dist.distance(mid, img_a) + vgg_dist.distance(mid, img_b)) / total

p10_rows = []
for k, (cat_a, cat_b, *_ ) in enumerate(p2_pairs):
    if cat_a not in category_data or cat_b not in category_data:
        continue
    if cat_a not in shuffle_score or cat_b not in shuffle_score:
        continue
    d = {"blob_a_gen": category_data[cat_a]["blob_a_gen"],
         "bf_a": category_data[cat_a]["bf_a"], "si_a": category_data[cat_a]["si_a"],
         "bf_b": category_data[cat_b]["bf_a"], "si_b": category_data[cat_b]["si_a"]}
    with torch.no_grad():
        img_a = decode_interp(category_data[cat_a]["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(category_data[cat_b]["blob_a_gen"], d["bf_b"], d["si_b"])
        total = vgg_dist.distance(img_a, img_b)
    if total < 1e-6:
        continue
    det_lin = midpoint_detour(d, img_a, img_b, total, "linear")
    det_ot  = midpoint_detour(d, img_a, img_b, total, "ot")
    margin = det_lin - det_ot
    ss = max(shuffle_score[cat_a], shuffle_score[cat_b])
    p10_rows.append((f"{cat_a}_vs_{cat_b}", ss, margin, det_lin, det_ot))
    print(f"[{k+1}/{len(p2_pairs)}] {cat_a+'_vs_'+cat_b:32s} "
          f"shuf={ss:.2f}  det_lin={det_lin:.2f}  det_ot={det_ot:.2f}  margin={margin:+.3f}")

ss = np.array([r[1] for r in p10_rows]); mg = np.array([r[2] for r in p10_rows])
sp = spearmanr(ss, mg); pe = pearsonr(ss, mg)
print(f"\n{len(p10_rows)} pairs | margin = detour(linear) - detour(OT), positive = OT wins")
print(f"OT wins (margin>0) on {int((mg > 0).sum())}/{len(mg)} pairs")
print(f"shuffle-shift vs margin: spearman r={sp.statistic:+.3f} (p={sp.pvalue:.1e}), "
      f"pearson r={pe.statistic:+.3f}")
print("prediction: NEGATIVE (arrangement-dependent textures => OT helps less)")

# same correlation excluding zigzag pairs (anomaly family)
mask = np.array(["zigzagged" not in r[0] for r in p10_rows])
if mask.sum() > 4:
    sp2 = spearmanr(ss[mask], mg[mask])
    print(f"excluding zigzag pairs:  spearman r={sp2.statistic:+.3f} (p={sp2.pvalue:.1e})")

plt.figure(figsize=(5.5, 4))
colors = ["#c0392b" if "zigzagged" in r[0] else "#2980b9" for r in p10_rows]
plt.scatter(ss, mg, s=28, alpha=0.75, c=colors)
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("pair shuffle-shift (max of endpoints)")
plt.ylabel("OT margin (VGG detour, linear − OT)")
plt.title("Link 2: H1 score vs OT advantage  (red = zigzag pairs)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p10_h1_predicts_margin.png"), dpi=150)
plt.show()

In [ ]:
# ============================================================
# Cell P11: Link 1 — zigzag mechanism: proper multimodality + structure stats
# Replaces the broken P7 instrument. Prediction (H3-mechanism): zigzagged tops
# the multimodality ranking. If it doesn't, the anomaly is real but H3 isn't
# the mechanism — and the structure stats below are the alternative clues.
# ============================================================
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

def multimodality_stats(cat, n=3000, n_pcs=8):
    sm = build_style_maps(category_data[cat]["bf_a"], category_data[cat]["si_a"], FMS)[largest][0]
    Ffull = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
    F_ = Ffull[torch.randperm(Ffull.shape[0])[:n]].cpu().numpy()
    Z = PCA(n_components=n_pcs).fit_transform(F_)

    # (a) GMM evidence for 2 modes vs 1 (higher = more bimodal)
    bic1 = GaussianMixture(1, covariance_type="full", random_state=0).fit(Z).bic(Z)
    bic2 = GaussianMixture(2, covariance_type="full", random_state=0).fit(Z).bic(Z)
    delta_bic = (bic1 - bic2) / n            # per-sample; >0 favors 2 modes

    # (b) dip-style stat on PC1: valley depth of the histogram between its two best peaks
    h, edges = np.histogram(Z[:, 0], bins=48, density=True)
    hs = np.convolve(h, np.ones(3) / 3, mode="same")
    peaks = [i for i in range(1, len(hs) - 1) if hs[i] > hs[i-1] and hs[i] > hs[i+1]]
    if len(peaks) >= 2:
        p1, p2 = sorted(sorted(peaks, key=lambda i: -hs[i])[:2])
        valley = 1 - hs[p1:p2+1].min() / (min(hs[p1], hs[p2]) + 1e-12)   # 0=unimodal, ->1=deep valley
    else:
        valley = 0.0

    # (c) anisotropy + heavy-tailedness (alternative mechanisms)
    ev = PCA(n_components=min(16, F_.shape[1])).fit(F_).explained_variance_ratio_
    anis = ev[0] / (ev[1] + 1e-12)
    r = np.linalg.norm(F_ - F_.mean(0), axis=1)
    kurt = float(((r - r.mean())**4).mean() / (r.var()**2 + 1e-12))
    return delta_bic, valley, anis, kurt

p11 = {}
for cat in category_data:
    p11[cat] = multimodality_stats(cat)

print(f"{'category':18s}{'dBIC/n':>9s}{'valley':>8s}{'anis':>7s}{'kurt':>7s}")
for cat, (db, vl, an, ku) in sorted(p11.items(), key=lambda kv: -kv[1][1]):
    flag = "  <-- zigzagged" if cat == "zigzagged" else ""
    print(f"{cat:18s}{db:9.3f}{vl:8.3f}{an:7.2f}{ku:7.2f}{flag}")

zz_rank_valley = sorted(p11, key=lambda c: -p11[c][1]).index("zigzagged") + 1
zz_rank_bic = sorted(p11, key=lambda c: -p11[c][0]).index("zigzagged") + 1
print(f"\nzigzagged rank: valley #{zz_rank_valley}/{len(p11)}, dBIC #{zz_rank_bic}/{len(p11)}")

# correlate each stat against exact-OT drop, WITH and WITHOUT zigzag pairs
from scipy.stats import spearmanr as _sp
names = ["dBIC/n", "valley", "anisotropy", "kurtosis"]
print(f"\n{'stat':12s}{'r_all':>8s}{'p_all':>10s}{'r_noZZ':>8s}{'p_noZZ':>10s}   (prediction: valley/dBIC negative)")
for si, name in enumerate(names):
    xs, ys, xs2, ys2 = [], [], [], []
    for r in p2_results:
        ca, cb = r["pair"].split("_vs_")
        if ca in p11 and cb in p11:
            v = max(p11[ca][si], p11[cb][si])
            xs.append(v); ys.append(r["exact_mid_over_chord"])
            if "zigzagged" not in (ca, cb):
                xs2.append(v); ys2.append(r["exact_mid_over_chord"])
    s1, s2 = _sp(xs, ys), _sp(xs2, ys2)
    print(f"{name:12s}{s1.statistic:+8.3f}{s1.pvalue:10.1e}{s2.statistic:+8.3f}{s2.pvalue:10.1e}")

In [ ]:
# ============================================================
# Cell P12: Control — P6 block-size sweep (does grid's mid-pack rank
# survive when the shuffle block is smaller than its period?)
# ============================================================
P12_CATS = ["chequered", "grid", "striped", "zigzagged", "swirly", "flecked", "gauzy", "woven"]
P12_BLOCKS = [1, 2, 4, 8]

print(f"{'category':14s}" + "".join(f"{'b='+str(b):>8s}" for b in P12_BLOCKS))
p12 = {}
for cat in P12_CATS:
    if cat not in category_data:
        continue
    with torch.no_grad():
        orig = decode_original(cat)
        vals = []
        for b in P12_BLOCKS:
            shuf = decode_shuffled(cat, b)
            vals.append(vgg_dist.distance(shuf, orig) / inter_scale)
    p12[cat] = vals
    print(f"{cat:14s}" + "".join(f"{v:8.3f}" for v in vals))

print("\nreading: if chequered/grid/striped JUMP at b=1-2 while flecked/gauzy stay flat,")
print("their mid-pack rank at b=4 was a block-size artifact and the refined H1 statement")
print("('long-range coherence, not periodicity') needs a caveat. if they stay low even")
print("at b=1, local periodicity genuinely survives marginal description and the refined")
print("statement stands as-is.")

In [ ]:
# ============================================================
# Cell P13: rho* — alignment under the OPTIMAL coupling
# (a) verify exact_mid_over_chord ≈ sqrt((1+rho*)/2) across pairs (algebra check)
# (b) rank pairs by rho*: zigzag pairs should be the low-rho* family
# Requires exact_ot_targets (P1), p2_pairs, category_data.
# ============================================================
from scipy.stats import spearmanr

p13 = []
for cat_a, cat_b, *_ in p2_pairs:
    if cat_a not in category_data or cat_b not in category_data:
        continue
    sm_a = build_style_maps(category_data[cat_a]["bf_a"], category_data[cat_a]["si_a"], FMS)[largest][0]
    sm_b = build_style_maps(category_data[cat_b]["bf_a"], category_data[cat_b]["si_b" if False else "si_a"], FMS)[largest][0]
    Xf = sm_a.permute(1, 2, 0).reshape(-1, sm_a.shape[0])
    Yf = sm_b.permute(1, 2, 0).reshape(-1, sm_b.shape[0])
    n = 1500
    X = Xf[torch.randperm(Xf.shape[0])[:n]]
    Y = Yf[torch.randperm(Yf.shape[0])[:n]]
    T = exact_ot_targets(X, Y)
    Xc, Tc = X - X.mean(0), T - T.mean(0)
    rho_star = ((Xc * Tc).sum(1).mean() /
                (Xc.norm(dim=1).mean() * Tc.norm(dim=1).mean() + 1e-12)).item()
    mid = centered_amplitude(0.5 * X + 0.5 * T)
    chord = (centered_amplitude(X) + centered_amplitude(T)) / 2
    p13.append((f"{cat_a}_vs_{cat_b}", rho_star,
                mid / chord, float(np.sqrt((1 + rho_star) / 2))))

p13.sort(key=lambda r: r[1])
print(f"{'pair':32s}{'rho*':>7s}{'obs':>7s}{'pred':>7s}")
for name, rs, obs, pred in p13:
    flag = "  <-- zigzag" if "zigzagged" in name else ""
    print(f"{name:32s}{rs:7.3f}{obs:7.3f}{pred:7.3f}{flag}")

obs = np.array([r[2] for r in p13]); pred = np.array([r[3] for r in p13])
print(f"\nalgebra check |obs - pred| mean: {np.abs(obs - pred).mean():.4f}  (should be tiny)")
zz = [r[1] for r in p13 if "zigzagged" in r[0]]
nz = [r[1] for r in p13 if "zigzagged" not in r[0]]
print(f"rho*: zigzag pairs {np.mean(zz):.3f} ± {np.std(zz):.3f}  vs others {np.mean(nz):.3f} ± {np.std(nz):.3f}")
print("if zigzag rho* is clearly lowest: anomaly = low optimal-coupling alignment;")
print("mechanism question becomes geometric (support shape), with modality ruled out by P11")

In [ ]:
# ============================================================
# Cell P13b: corrected rho* (RMS/second-moment definitions — exact identity)
# and decomposition: is zigzag's low rho* scale mismatch or angular misalignment?
# ============================================================
p13b = []
for cat_a, cat_b, *_ in p2_pairs:
    if cat_a not in category_data or cat_b not in category_data:
        continue
    sm_a = build_style_maps(category_data[cat_a]["bf_a"], category_data[cat_a]["si_a"], FMS)[largest][0]
    sm_b = build_style_maps(category_data[cat_b]["bf_a"], category_data[cat_b]["si_a"], FMS)[largest][0]
    Xf = sm_a.permute(1, 2, 0).reshape(-1, sm_a.shape[0])
    Yf = sm_b.permute(1, 2, 0).reshape(-1, sm_b.shape[0])
    n = 1500
    X = Xf[torch.randperm(Xf.shape[0])[:n]]
    Y = Yf[torch.randperm(Yf.shape[0])[:n]]
    T = exact_ot_targets(X, Y)
    Xc, Tc = X - X.mean(0), T - T.mean(0)

    a0 = Xc.pow(2).sum(1).mean().sqrt().item()          # RMS amplitude
    a1 = Tc.pow(2).sum(1).mean().sqrt().item()
    cross = (Xc * Tc).sum(1).mean().item()
    rho = cross / (a0 * a1)                              # proper correlation, |rho|<=1
    mid = (0.5 * Xc + 0.5 * Tc).pow(2).sum(1).mean().sqrt().item()
    pred = np.sqrt(a0**2 + a1**2 + 2 * rho * a0 * a1) / 2

    # decomposition: cosine alignment of paired displacements vs scale ratio
    cos_pair = ((Xc * Tc).sum(1) /
                (Xc.norm(dim=1) * Tc.norm(dim=1) + 1e-12)).mean().item()  # angular
    scale_ratio = min(a0, a1) / max(a0, a1)                                # scale
    # normalized transport cost: E|X-T|^2 relative to scales
    w2rel = (X - T).pow(2).sum(1).mean().item() / (a0**2 + a1**2)

    p13b.append((f"{cat_a}_vs_{cat_b}", rho, mid, pred, cos_pair, scale_ratio, w2rel))

p13b.sort(key=lambda r: r[1])
print(f"{'pair':32s}{'rho*':>7s}{'mid':>7s}{'pred':>7s}{'cos':>7s}{'scale':>7s}{'W2rel':>7s}")
for name, rho, mid, pred, cs, sr, w2 in p13b:
    flag = "  <-- zigzag" if "zigzagged" in name else ""
    print(f"{name:32s}{rho:7.3f}{mid:7.3f}{pred:7.3f}{cs:7.3f}{sr:7.3f}{w2:7.3f}{flag}")

err = np.array([abs(r[2] - r[3]) / r[3] for r in p13b])
print(f"\nexact identity check, mean relative |mid - pred|/pred: {err.mean():.2e}  (should be ~1e-7)")
zz = [r for r in p13b if "zigzagged" in r[0]]
nz = [r for r in p13b if "zigzagged" not in r[0]]
for label, grp in [("zigzag", zz), ("others", nz)]:
    print(f"{label:8s} rho*={np.mean([r[1] for r in grp]):.3f}±{np.std([r[1] for r in grp]):.3f}  "
          f"cos={np.mean([r[4] for r in grp]):.3f}  scale={np.mean([r[5] for r in grp]):.3f}  "
          f"W2rel={np.mean([r[6] for r in grp]):.3f}")
print("\nreading: if zigzag's gap shows in cos (angular) but not scale => support is")
print("rotated/curved relative to others (geometric H3 violation). if in scale => ")
print("amplitude mismatch story. W2rel high for zigzag either way (it's ~equivalent to low rho*).")

In [ ]:
# ============================================================
# Cell P14: is the anomaly the CATEGORY or the EXEMPLAR?
# Load 5 zigzagged images + 5 striped (control), compute rho*/cos/W2rel
# of each against a fixed partner (lined). Also check mean-offset direction.
# ============================================================
def load_category_exemplars(cat, k=5, seed=7):
    cat_dir = os.path.join(dtd_root, cat)
    imgs = sorted([f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))])
    random.seed(seed)
    picks = random.sample(imgs, min(k, len(imgs)))
    out = []
    for name in picks:
        try:
            it = load_texture(os.path.join(cat_dir, name))
            blob, bf, si = get_generator_inputs(it)
            out.append((name, bf, si))
        except Exception as e:
            print(f"  skip {name}: {e}")
    return out

def cloud_of(bf, si, n=1500):
    sm = build_style_maps(bf, si, FMS)[largest][0]
    Ff = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
    return Ff[torch.randperm(Ff.shape[0])[:n]]

def coupling_stats(X, Y):
    T = exact_ot_targets(X, Y)
    Xc, Tc = X - X.mean(0), T - T.mean(0)
    a0 = Xc.pow(2).sum(1).mean().sqrt().item()
    a1 = Tc.pow(2).sum(1).mean().sqrt().item()
    rho = (Xc * Tc).sum(1).mean().item() / (a0 * a1)
    cos = ((Xc * Tc).sum(1) / (Xc.norm(dim=1) * Tc.norm(dim=1) + 1e-12)).mean().item()
    w2rel = (X - T).pow(2).sum(1).mean().item() / (a0**2 + a1**2)
    mean_off = (X.mean(0) - Y.mean(0)).norm().item() / np.sqrt((a0**2 + a1**2) / 2)
    return rho, cos, w2rel, mean_off

partner = cloud_of(category_data["lined"]["bf_a"], category_data["lined"]["si_a"])
print(f"{'exemplar':34s}{'rho*':>7s}{'cos':>7s}{'W2rel':>8s}{'mean_off':>9s}")
p14 = {}
for cat in ["zigzagged", "striped"]:
    for name, bf, si in load_category_exemplars(cat):
        X = cloud_of(bf, si)
        stats = coupling_stats(X, partner)
        p14[f"{cat}/{name}"] = stats
        print(f"{cat + '/' + name:34s}{stats[0]:7.3f}{stats[1]:7.3f}{stats[2]:8.2f}{stats[3]:9.2f}")

# and the original exemplar for reference
X0 = cloud_of(category_data["zigzagged"]["bf_a"], category_data["zigzagged"]["si_a"])
s0 = coupling_stats(X0, partner)
print(f"{'zigzagged/ORIGINAL (cell 10 draw)':34s}{s0[0]:7.3f}{s0[1]:7.3f}{s0[2]:8.2f}{s0[3]:9.2f}")
print("\nreading: all zigzag exemplars at rho*~0.4-0.6 => category-level, H3-violation")
print("story stands. only the original => exemplar artifact; check that image visually")
print("and the story becomes 'a diagnostic that catches off-manifold encodings'.")

In [ ]:
# ============================================================
# Cell P15: Gap 1a — multi-exemplar rerun of the core Prop-2 numbers
# 3 exemplars per category for the 10 categories in the pair list;
# recomputes the scaffold from the multi-exemplar set; reports the spread
# of every headline number; screens all encodings with rho* (diagnostic
# in production — catches any other zigzagged_0032-style outliers).
# ============================================================
P15_CATS = ["chequered", "grid", "honeycombed", "meshed", "polka-dotted",
            "striped", "zigzagged", "lined", "woven", "knitted", "braided"]
P15_K = 3
P15_SEED = 123          # different from Cell 10's seed 42 on purpose

multi_data = {}
for cat in P15_CATS:
    cat_dir = os.path.join(dtd_root, cat)
    imgs = sorted([f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))])
    random.seed(P15_SEED + hash(cat) % 1000)
    for name in random.sample(imgs, min(P15_K, len(imgs))):
        try:
            it = load_texture(os.path.join(cat_dir, name))
            blob, bf, si = get_generator_inputs(it)
            multi_data[f"{cat}/{name}"] = {"cat": cat, "blob": blob, "bf": bf, "si": si}
        except Exception as e:
            print(f"  skip {cat}/{name}: {e}")
print(f"encoded {len(multi_data)} exemplars")

def cloud_of_entry(e, n=4096):
    sm = build_style_maps(e["bf"], e["si"], FMS)[largest][0]
    Ff = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
    idx = torch.randperm(Ff.shape[0], device=Ff.device)[:n]
    return Ff[idx], idx

# multi-exemplar scaffold: mean style map over ALL encodings, per position
with torch.no_grad():
    stack = torch.stack([build_style_maps(e["bf"], e["si"], FMS)[largest][0]
                         for e in multi_data.values()])
    scaffold = stack.mean(0)
sc_flat = scaffold.permute(1, 2, 0).reshape(-1, scaffold.shape[0])

def rms_amp(F):
    return (F - F.mean(0)).pow(2).sum(1).mean().sqrt().item()

def rms_rho(X, Y):
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    return (Xc * Yc).sum(1).mean().item() / (rms_amp(X) * rms_amp(Y) + 1e-12)

def rho_star_of(X, Y, n_exact=1500):
    Xe, Ye = X[:n_exact], Y[:n_exact]
    T = exact_ot_targets(Xe, Ye)
    return rms_rho(Xe, T), Xe, T

keys_by_cat = {}
for k, e in multi_data.items():
    keys_by_cat.setdefault(e["cat"], []).append(k)

cat_pairs = sorted({(a, b) for a, b, *_ in p2_pairs
                    if a in keys_by_cat and b in keys_by_cat})
p15_rows, p15_flags = [], []
for ca, cb in cat_pairs:
    for i in range(min(P15_K, len(keys_by_cat[ca]), len(keys_by_cat[cb]))):
        ka, kb = keys_by_cat[ca][i], keys_by_cat[cb][i]
        X, idx = cloud_of_entry(multi_data[ka])
        Y, _   = cloud_of_entry(multi_data[kb])
        # spatial coupling on residuals (scaffold removed at matched positions)
        Xr, Yr = X - sc_flat[idx], Y - sc_flat[idx]
        rho_res = rms_rho(Xr, Yr)
        mid_lin = rms_amp(0.5 * Xr + 0.5 * Yr)
        chord_lin = (rms_amp(Xr) + rms_amp(Yr)) / 2
        col_res = mid_lin / chord_lin
        pred_res = float(np.sqrt((1 + rho_res) / 2))
        # geodesic side + diagnostic
        rs, Xe, T = rho_star_of(X, Y)
        exact_mid = rms_amp(0.5 * Xe + 0.5 * T) / ((rms_amp(Xe) + rms_amp(T)) / 2)
        p15_rows.append((f"{ka} vs {kb}", rho_res, col_res, pred_res, rs, exact_mid))
        if rs < 0.8:
            p15_flags.append((ka, kb, rs))

rr = np.array([r[1] for r in p15_rows]); er = np.array([abs(r[2] - r[3]) for r in p15_rows])
rs_ = np.array([r[4] for r in p15_rows]); ex = np.array([r[5] for r in p15_rows])
ok = rs_ >= 0.8
print(f"\n{len(p15_rows)} exemplar-pairs measured, {int((~ok).sum())} flagged by rho* (<0.8)")
for ka, kb, rs in p15_flags:
    print(f"  FLAGGED: {ka} vs {kb}  rho*={rs:.3f}   <- inspect these encodings")
print("\n--- core numbers on rho*-clean pairs (v2 doc values in parens) ---")
print(f"residual rho:        {rr[ok].mean():+.3f} ± {rr[ok].std():.3f}   (-0.025)")
print(f"collapse |obs-pred|: {er[ok].mean():.4f} ± {er[ok].std():.4f}   (0.018)")
print(f"exact-OT mid/chord:  {ex[ok].mean():.3f} ± {ex[ok].std():.3f}   (0.96-1.01)")
print(f"rho* clean cluster:  {rs_[ok].mean():.3f} ± {rs_[ok].std():.3f}   (0.985)")
print("\nverdict: if the four numbers hold within spread, Prop 2 + Prop 1 signature are")
print("exemplar-robust and the v2 doc's single-exemplar caveat is discharged for the core.")

In [ ]:
# ============================================================
# Cell P16: Gap 1b — multi-exemplar H1 boundary (shuffle-shift)
# 5 exemplars for the boundary-defining categories, b=1 and b=4.
# The claim under test: 'long-range coherence, not periodicity'.
# ============================================================
P16_CATS = ["swirly", "spiralled", "braided", "interlaced",       # claimed H1-weak
            "chequered", "grid", "striped",                        # periodic controls
            "flecked", "gauzy"]                                    # claimed H1-strong
P16_K = 5

def shuffle_shift_for(bf, si, blob, b):
    sm = build_style_maps(bf, si, FMS)
    shuf = block_shuffle_map(sm[largest], b)
    smaps = {}
    for size in set(FMS):
        smaps[size] = shuf if size == largest else F.interpolate(
            shuf, size=size, mode="bilinear", align_corners=False)
    with torch.no_grad():
        orig = decode_with_style_maps(blob, sm)   # decode via full style-map path
        shf  = decode_with_style_maps(blob, smaps)
        return vgg_dist.distance(shf, orig) / inter_scale

p16 = {}
print(f"{'category':14s}{'b=1 mean±sd':>16s}{'b=4 mean±sd':>16s}{'n':>4s}")
for cat in P16_CATS:
    vals1, vals4 = [], []
    for name, bf, si in load_category_exemplars(cat, k=P16_K, seed=11):
        it_blob, _, _ = None, None, None
        try:
            img = load_texture(os.path.join(dtd_root, cat, name))
            blob, bf2, si2 = get_generator_inputs(img)
            vals1.append(shuffle_shift_for(bf2, si2, blob, 1))
            vals4.append(shuffle_shift_for(bf2, si2, blob, 4))
        except Exception as e:
            print(f"  skip {cat}/{name}: {e}")
    if vals1:
        p16[cat] = (np.mean(vals1), np.std(vals1), np.mean(vals4), np.std(vals4))
        print(f"{cat:14s}{p16[cat][0]:8.2f}±{p16[cat][1]:.2f}   "
              f"{p16[cat][2]:8.2f}±{p16[cat][3]:.2f} {len(vals1):4d}")

flow = [c for c in ["swirly", "spiralled", "braided", "interlaced"] if c in p16]
peri = [c for c in ["chequered", "grid", "striped"] if c in p16]
stoc = [c for c in ["flecked", "gauzy"] if c in p16]
print(f"\nflow-structure mean (b=1):  {np.mean([p16[c][0] for c in flow]):.2f}")
print(f"periodic mean (b=1):        {np.mean([p16[c][0] for c in peri]):.2f}")
print(f"stochastic mean (b=1):      {np.mean([p16[c][0] for c in stoc]):.2f}")
print("claim holds if: flow >> periodic ~ moderate > stochastic, with category sd")
print("small relative to the between-group gaps (i.e., not driven by single exemplars)")

In [ ]:
# ============================================================
# Cell P17: Gap 2 — does OT-sym fix the balance while keeping coherence?
# Uses YOUR interpolate_ot_sym(d, eta) if defined; else a built-in two-sided
# fallback (same midpoint marginal decoded from BOTH layouts) that tests the
# diagnosis but is NOT your method — results then motivate, not validate, OT-sym.
# ============================================================
HAVE_OTSYM = "interpolate_ot_sym" in globals()
print(f"using {'YOUR interpolate_ot_sym' if HAVE_OTSYM else 'built-in two-sided fallback'}")

def two_sided_decode(d, eta):
    """Fallback: A-side and B-side one-sided OT midpoints (same marginal,
    different layouts). Returns both decodes."""
    dA = d
    dB = {"blob_a_gen": d["blob_b_gen"], "bf_a": d["bf_b"], "si_a": d["si_b"],
          "bf_b": d["bf_a"], "si_b": d["si_a"]}
    fn = lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100)
    return (decode_with_transport(dA, eta, fn),
            decode_with_transport(dB, 1 - eta, fn))

P17_PAIRS = [("chequered", "grid"), ("woven", "knitted"),
             ("polka-dotted", "striped"), ("honeycombed", "lined")]

print(f"\n{'pair':26s}{'bal 1-sided':>12s}{'bal sym':>9s}{'det 1-sided':>12s}{'det sym':>9s}")
fig, axes = plt.subplots(len(P17_PAIRS), 4, figsize=(13, 3.3 * len(P17_PAIRS)))
for row, (ca, cb) in enumerate(P17_PAIRS):
    d = {"blob_a_gen": category_data[ca]["blob_a_gen"],
         "blob_b_gen": category_data[cb]["blob_a_gen"],
         "bf_a": category_data[ca]["bf_a"], "si_a": category_data[ca]["si_a"],
         "bf_b": category_data[cb]["bf_a"], "si_b": category_data[cb]["si_a"]}
    with torch.no_grad():
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        total = vgg_dist.distance(img_a, img_b)
        one_sided = decode_with_transport(d, 0.5,
            lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100))
        if HAVE_OTSYM:
            sym = interpolate_ot_sym(d, 0.5)
        else:
            sA, sB = two_sided_decode(d, 0.5)
            sym = sA  # display A-side; metrics below use both
    def bal_det(img):
        dA_ = vgg_dist.distance(img, img_a); dB_ = vgg_dist.distance(img, img_b)
        return (dA_ - dB_) / total, (dA_ + dB_) / total
    b1, det1 = bal_det(one_sided)
    if HAVE_OTSYM:
        bs, dets = bal_det(sym)
    else:
        (bsA, detA_), (bsB, detB_) = bal_det(sA), bal_det(sB)
        bs, dets = (bsA + bsB) / 2, (detA_ + detB_) / 2   # fallback: average of sides
    print(f"{ca+'_vs_'+cb:26s}{b1:+12.2f}{bs:+9.2f}{det1:12.2f}{dets:9.2f}")
    for col, (im, ttl) in enumerate([(img_a, "A"), (one_sided, f"1-sided b={b1:+.2f}"),
                                     (sym, f"{'OT-sym' if HAVE_OTSYM else 'sym-fb'} b={bs:+.2f}"),
                                     (img_b, "B")]):
        ax = axes[row, col]; ax.imshow(tensor_to_image(im)); ax.axis("off")
        ax.set_title(ttl, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p17_otsym_balance.png"), dpi=150)
plt.show()
print("\nsuccess criterion: |bal sym| << |bal 1-sided| with detour not worse and")
print("midpoints visually coherent (single texture, no ghosting)")

In [ ]:
# ============================================================
# Cell P18: Gap 3 — direct H3 (ellipticity) test: full OT vs Gaussian OT (WCT)
# If clouds are elliptical, the Gaussian (Bures) map ~ the true Brenier map,
# so their midpoint clouds should nearly coincide. Gap = W2(full-mid, gauss-mid)
# relative to W2(A, B). Small uniformly => H3's sufficient condition supported.
# ============================================================
def gaussian_ot_map(X, Y, reg=1e-4):
    """Bures/WCT map: T(x) = mY + A(x - mX)."""
    mX, mY = X.mean(0), Y.mean(0)
    Xc, Yc = (X - mX), (Y - mY)
    d = X.shape[1]
    SX = Xc.T @ Xc / X.shape[0] + reg * torch.eye(d, device=X.device)
    SY = Yc.T @ Yc / Y.shape[0] + reg * torch.eye(d, device=X.device)
    def sqrtm(S):
        ev, V = torch.linalg.eigh(S)
        return V @ torch.diag(ev.clamp_min(0).sqrt()) @ V.T
    SXh = sqrtm(SX); SXhi = torch.linalg.inv(SXh)
    A = SXhi @ sqrtm(SXh @ SY @ SXh) @ SXhi
    return mY + (X - mX) @ A.T

def w2_between_clouds(P, Q, n=1000):
    Pi = P[torch.randperm(P.shape[0])[:n]]
    Qi = Q[torch.randperm(Q.shape[0])[:n]]
    C = torch.cdist(Pi, Qi).pow(2).cpu().numpy()
    r, c = linear_sum_assignment(C)
    return float(np.sqrt(C[r, c].mean()))

print(f"{'pair':32s}{'gap/W2(A,B)':>12s}")
p18 = []
for ca, cb, *_ in p2_pairs:
    if ca not in category_data or cb not in category_data:
        continue
    sm_a = build_style_maps(category_data[ca]["bf_a"], category_data[ca]["si_a"], FMS)[largest][0]
    sm_b = build_style_maps(category_data[cb]["bf_a"], category_data[cb]["si_a"], FMS)[largest][0]
    Xf = sm_a.permute(1, 2, 0).reshape(-1, sm_a.shape[0])
    Yf = sm_b.permute(1, 2, 0).reshape(-1, sm_b.shape[0])
    n = 1500
    X = Xf[torch.randperm(Xf.shape[0])[:n]]
    Y = Yf[torch.randperm(Yf.shape[0])[:n]]
    T_full = exact_ot_targets(X, Y)
    with torch.no_grad():
        T_gauss = gaussian_ot_map(X, Y)
    mid_full  = 0.5 * X + 0.5 * T_full
    mid_gauss = 0.5 * X + 0.5 * T_gauss
    gap = w2_between_clouds(mid_full, mid_gauss)
    scale = w2_between_clouds(X, Y)
    rel = gap / (scale + 1e-12)
    flag = "  <-- zigzag(0032)" if "zigzagged" in (ca, cb) else ""
    p18.append((f"{ca}_vs_{cb}", rel))
    print(f"{ca+'_vs_'+cb:32s}{rel:12.3f}{flag}")

rels = np.array([r[1] for r in p18])
mask = np.array(["zigzagged" not in r[0] for r in p18])
print(f"\nrelative gap: all pairs {rels.mean():.3f} ± {rels.std():.3f}")
print(f"excluding 0032 pairs:   {rels[mask].mean():.3f} ± {rels[mask].std():.3f}")
print("reading: uniformly small (~<0.15) on clean pairs => the Gaussian map is a good")
print("proxy for the Brenier map => ellipticity => H3's sufficient condition supported;")
print("large on 0032 pairs specifically => consistent with the off-manifold diagnosis.")

In [ ]:
ref = cloud_of_entry(multi_data["chequered/" + keys_by_cat["chequered"][0].split("/")[1]])[0] \
      if False else cloud_of_entry(list(multi_data.values())[0])[0]  # any clean third texture
for k in ["woven/woven_0117.jpg", "knitted/knitted_0123.jpg"]:
    X, _ = cloud_of_entry(multi_data[k])
    rs, _, _ = rho_star_of(X, ref)
    print(k, "rho* vs third texture:", round(rs, 3))
# the one with low rho* against the third party is the corrupted encoding; decode and look at it

In [ ]:
# ============================================================
# Cell P19: quick — characterize the knitted_0123 corruption
# ============================================================
e = multi_data["knitted/knitted_0123.jpg"]
Xbad, _ = cloud_of_entry(e)
egood = multi_data[[k for k in keys_by_cat["knitted"] if "0123" not in k][0]]
Xgood, _ = cloud_of_entry(egood)
ref, _ = cloud_of_entry(multi_data[keys_by_cat["striped"][0]])

for name, X in [("knitted_0123 (bad)", Xbad), ("knitted (good)", Xgood)]:
    mean_off = (X.mean(0) - ref.mean(0)).norm().item() / rms_amp(ref)
    print(f"{name:22s} rms_amp={rms_amp(X):7.3f}  mean_offset_vs_ref={mean_off:7.2f}")
with torch.no_grad():
    img = decode_with_style_maps(e["blob"], build_style_maps(e["bf"], e["si"], FMS))
plt.figure(figsize=(3.5, 3.5)); plt.imshow(tensor_to_image(img)); plt.axis("off")
plt.title("knitted_0123 reconstruction"); plt.show()
# also open the raw dtd image side by side and note what's unusual about it

In [ ]:
# ============================================================
# Cell P20: THE H3 test — round-trip realizability of the geodesic
# For t in {0.25, 0.5, 0.75}: build geodesic cloud, decode, RE-ENCODE the
# decoded image, measure how far the re-encoded cloud is from the intended one.
# Baselines: (i) round-trip error of the ENDPOINTS themselves (encoder noise
# floor); (ii) same test for the LINEAR blend (off-manifold reference).
# on-manifold verdict: geodesic round-trip error ~ endpoint floor << linear's.
# ============================================================
def reencode(img_tensor):
    """image -> style-map cloud (through the encoder)."""
    _, bf, si = get_generator_inputs(img_tensor)
    sm = build_style_maps(bf, si, FMS)[largest][0]
    return sm.permute(1, 2, 0).reshape(-1, sm.shape[0])

def w2rel_clouds(P, Q, n=1000):
    Pi = P[torch.randperm(P.shape[0])[:n]]; Qi = Q[torch.randperm(Q.shape[0])[:n]]
    C = torch.cdist(Pi, Qi).pow(2).cpu().numpy()
    r, c = linear_sum_assignment(C)
    return float(np.sqrt(C[r, c].mean())) / (0.5 * (rms_amp(P) + rms_amp(Q)) + 1e-12)

P20_PAIRS = [("chequered", "grid"), ("woven", "braided"),
             ("honeycombed", "lined"), ("polka-dotted", "striped")]
P20_TS = [0.25, 0.5, 0.75]

print(f"{'pair':26s}{'t':>5s}{'geo_rt':>8s}{'lin_rt':>8s}{'floorA':>8s}{'floorB':>8s}")
p20 = []
for ca, cb in P20_PAIRS:
    d = {"blob_a_gen": category_data[ca]["blob_a_gen"],
         "bf_a": category_data[ca]["bf_a"], "si_a": category_data[ca]["si_a"],
         "bf_b": category_data[cb]["bf_a"], "si_b": category_data[cb]["si_a"]}
    # encoder noise floor: decode each endpoint, re-encode, distance to its own cloud
    with torch.no_grad():
        sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
        sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
        Xa = sm_a[largest][0].permute(1, 2, 0).reshape(-1, sm_a[largest].shape[1])
        Xb = sm_b[largest][0].permute(1, 2, 0).reshape(-1, sm_b[largest].shape[1])
        floorA = w2rel_clouds(reencode(decode_with_style_maps(d["blob_a_gen"], sm_a)), Xa)
        floorB = w2rel_clouds(reencode(decode_with_style_maps(d["blob_a_gen"], sm_b)), Xb)
    for t in P20_TS:
        with torch.no_grad():
            geo_img = decode_with_transport(d, t,
                lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100))
            lin_img = interpolate_linear_pixel(d, t)
            # intended geodesic cloud at t (recompute transport on a subsample)
            n = 1500
            Xs = Xa[torch.randperm(Xa.shape[0])[:n]]
            Ys = Xb[torch.randperm(Xb.shape[0])[:n]]
            T = exact_ot_targets(Xs, Ys)
            mu_geo = (1 - t) * Xs + t * T
            mu_lin = (1 - t) * Xs + t * Ys
            geo_rt = w2rel_clouds(reencode(geo_img), mu_geo)
            lin_rt = w2rel_clouds(reencode(lin_img), mu_lin)
        p20.append((f"{ca}_vs_{cb}", t, geo_rt, lin_rt, floorA, floorB))
        print(f"{ca+'_vs_'+cb:26s}{t:5.2f}{geo_rt:8.3f}{lin_rt:8.3f}{floorA:8.3f}{floorB:8.3f}")

g = np.array([r[2] for r in p20]); l = np.array([r[3] for r in p20])
f = np.array([(r[4] + r[5]) / 2 for r in p20])
print(f"\ngeodesic round-trip: {g.mean():.3f} ± {g.std():.3f}")
print(f"linear round-trip:   {l.mean():.3f} ± {l.std():.3f}")
print(f"endpoint floor:      {f.mean():.3f} ± {f.std():.3f}")
print("verdict: geo ~ floor => geodesic marginals are realizable textures (operational")
print("H2+H3, no ellipticity needed). geo << lin => linear's marginals are NOT preserved")
print("through the pipeline — the encoder itself rejects them (renormalization, §6).")

In [ ]:
# ============================================================
# Cell P21: geodesic self-consistency (internal coherence of the family)
# A constant-speed geodesic satisfies: midpoint(A, mid(A,B)) = quarter-point(A,B).
# Test in feature space: build both clouds, measure their relative W2.
# Baseline: same construction under the linear/spatial coupling.
# ============================================================
print(f"{'pair':26s}{'geo_inconsist':>14s}{'lin_inconsist':>14s}")
p21 = []
for ca, cb in P20_PAIRS:
    sm_a = build_style_maps(category_data[ca]["bf_a"], category_data[ca]["si_a"], FMS)[largest][0]
    sm_b = build_style_maps(category_data[cb]["bf_a"], category_data[cb]["si_a"], FMS)[largest][0]
    Xf = sm_a.permute(1, 2, 0).reshape(-1, sm_a.shape[0])
    Yf = sm_b.permute(1, 2, 0).reshape(-1, sm_b.shape[0])
    n = 1200
    X = Xf[torch.randperm(Xf.shape[0])[:n]]; Y = Yf[torch.randperm(Yf.shape[0])[:n]]
    # geodesic: direct quarter point vs two-step
    T_ab = exact_ot_targets(X, Y)
    q_direct = 0.75 * X + 0.25 * T_ab                 # t=0.25 on A->B
    mid = 0.5 * X + 0.5 * T_ab
    T_am = exact_ot_targets(X, mid)
    q_twostep = 0.5 * X + 0.5 * T_am                  # midpoint of A and mid
    geo_inc = w2rel_clouds(q_direct, q_twostep)
    # linear/spatial analogue (same-index coupling both steps)
    q_direct_l = 0.75 * X + 0.25 * Y
    mid_l = 0.5 * X + 0.5 * Y
    q_twostep_l = 0.5 * X + 0.5 * mid_l
    lin_inc = w2rel_clouds(q_direct_l, q_twostep_l)
    p21.append((geo_inc, lin_inc))
    print(f"{ca+'_vs_'+cb:26s}{geo_inc:14.3f}{lin_inc:14.3f}")

gi = np.array([r[0] for r in p21])
print(f"\ngeodesic inconsistency: {gi.mean():.3f} ± {gi.std():.3f}")
print("small (~<0.05) => the OT interpolation family is internally consistent")
print("(constant-speed geodesic property holds empirically, strengthening Prop 1")
print("beyond the amplitude signature). note the linear column is ~0 by construction")
print("(same-index coupling is trivially consistent) — it's a sanity bound, not a rival.")

In [ ]:
# does the flagged-but-well-reconstructed encoding actually interpolate badly?
e123 = multi_data["knitted/knitted_0123.jpg"]
d = {"blob_a_gen": category_data["woven"]["blob_a_gen"],
     "bf_a": category_data["woven"]["bf_a"], "si_a": category_data["woven"]["si_a"],
     "bf_b": e123["bf"], "si_b": e123["si"]}
with torch.no_grad():
    mid = decode_with_transport(d, 0.5,
        lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100))
    # round-trip the geodesic midpoint, same test as P20
    Xa = build_style_maps(d["bf_a"], d["si_a"], FMS)[largest][0].permute(1,2,0).reshape(-1, 32 if False else build_style_maps(d["bf_a"], d["si_a"], FMS)[largest].shape[1])
    Xb = build_style_maps(d["bf_b"], d["si_b"], FMS)[largest][0].permute(1,2,0).reshape(-1, Xa.shape[1])
    n = 1500
    Xs = Xa[torch.randperm(Xa.shape[0])[:n]]; Ys = Xb[torch.randperm(Xb.shape[0])[:n]]
    T = exact_ot_targets(Xs, Ys)
    rt = w2rel_clouds(reencode(mid), 0.5 * Xs + 0.5 * T)
plt.figure(figsize=(3.5, 3.5)); plt.imshow(tensor_to_image(mid)); plt.axis("off")
plt.title(f"woven <-> knitted_0123 midpoint | round-trip {rt:.3f}"); plt.show()
print("P20 floor was ~0.12: if rt >> that, the flag was justified despite the clean")
print("reconstruction; if rt ~ 0.12 and the image looks coherent, 0123 is a false")
print("positive and the diagnostic claim narrows to 'flags location outliers,")
print("which included the one genuine encoder failure found'")

In [ ]:
# ============================================================
# Cell P22: P20-clean — round-trip realizability, deconfounded
# Fixes: (1) SAME sampled points + SAME transport targets construct the
# intermediate AND define its intended marginal; (2) floorB uses B's own blob;
# (3) symmetric A-side and B-side decodes; (4) stratified pair sample.
# Transport hierarchy is explicit: Hungarian reference throughout (the object
# the theorem speaks about), production Sinkhorn reported separately.
# ============================================================
random.seed(202)
P22_TS = [0.25, 0.5, 0.75]
P22_N = 12   # stratified: sample across the full all_pairs list, not its head
_pool = [p for p in all_pairs if p[0] in category_data and p[1] in category_data]
random.shuffle(_pool)
P22_PAIRS = [(a, b) for a, b, *_ in _pool[:P22_N]]

def decode_from_targets(blob, src_flat, all_targets, t, C_ch, H, W):
    """Decode the blend (1-t)src + t*targets — targets supplied, not recomputed."""
    interp = ((1 - t) * src_flat + t * all_targets).reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
    smaps = {}
    for size in set(FMS):
        smaps[size] = interp if size == largest else F.interpolate(
            interp, size=size, mode="bilinear", align_corners=False)
    return decode_with_style_maps(blob, smaps)

def side_run(blob_src, sm_src, sm_tgt, ts, n_sub=4096, n_exact=2000):
    """One direction: source cloud -> target cloud. Returns per-t round-trips
    where intended marginal and decoded image share the SAME transport targets."""
    src = sm_src[largest][0]; tgt = sm_tgt[largest][0]
    C_ch, H, W = src.shape
    src_flat = src.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = tgt.permute(1, 2, 0).reshape(-1, C_ch)
    N = src_flat.shape[0]
    idx_s = torch.randperm(N, device=src_flat.device)[:n_sub]
    idx_t = torch.randperm(N, device=src_flat.device)[:n_sub]
    Xs, Yt = src_flat[idx_s], tgt_flat[idx_t]
    # Hungarian on a sub-block, NN-extended to the n_sub set (hierarchy: reference)
    Xe, Ye = Xs[:n_exact], Yt[:n_exact]
    Te = exact_ot_targets(Xe, Ye)
    nn_sub = torch.cdist(Xs, Xe).argmin(dim=1)
    T_sub = Te[nn_sub]                       # targets for the n_sub cloud
    nn_full = torch.cdist(src_flat, Xs).argmin(dim=1)
    T_full = T_sub[nn_full]                  # targets for every pixel (same map!)
    out = []
    for t in ts:
        mu_intended = (1 - t) * Xs + t * T_sub          # defined by the SAME targets
        with torch.no_grad():
            img = decode_from_targets(blob_src, src_flat, T_full, t, C_ch, H, W)
            rt = w2rel_clouds(reencode(img), mu_intended)
        out.append(rt)
    return out

print(f"{'pair':30s}{'side':>5s}" + "".join(f"{'t='+str(t):>8s}" for t in P22_TS)
      + f"{'floor':>8s}")
geo_all, floor_all = [], []
for ca, cb in P22_PAIRS:
    ea, eb = category_data[ca], category_data[cb]
    sm_a = build_style_maps(ea["bf_a"], ea["si_a"], FMS)
    sm_b = build_style_maps(eb["bf_a"], eb["si_a"], FMS)
    # noise floors, each texture through its OWN blob (fixes floorB bug)
    with torch.no_grad():
        Xa = sm_a[largest][0].permute(1, 2, 0).reshape(-1, sm_a[largest].shape[1])
        Xb = sm_b[largest][0].permute(1, 2, 0).reshape(-1, sm_b[largest].shape[1])
        fA = w2rel_clouds(reencode(decode_with_style_maps(ea["blob_a_gen"], sm_a)), Xa)
        fB = w2rel_clouds(reencode(decode_with_style_maps(eb["blob_a_gen"], sm_b)), Xb)
    rtA = side_run(ea["blob_a_gen"], sm_a, sm_b, P22_TS)   # A -> B, A's blob
    rtB = side_run(eb["blob_a_gen"], sm_b, sm_a, P22_TS)   # B -> A, B's blob
    for side, rts, fl in [("A", rtA, fA), ("B", rtB, fB)]:
        print(f"{ca+'_vs_'+cb:30s}{side:>5s}" + "".join(f"{r:8.3f}" for r in rts)
              + f"{fl:8.3f}")
        geo_all += rts; floor_all.append(fl)

g = np.array(geo_all); f = np.array(floor_all)
print(f"\ngeodesic round-trip (n={len(g)}): {g.mean():.3f} ± {g.std():.3f}")
print(f"endpoint floor (n={len(f)}):      {f.mean():.3f} ± {f.std():.3f}")
print("\nverdict: geo ~ floor on the SAME transport, both sides, stratified pairs =>")
print("'operationally verified' stands with the confound removed. materially above")
print("floor => the earlier result was partly the epsilon/subsample mismatch,")
print("and the doc's H3 row downgrades to 'supported'.")

In [ ]:
# ============================================================
# Cell P23: H1 shuffle test, H2'-deconfounded
# The blob/layout side channel is neutralized by decoding each condition under
# TWO foreign blobs (from unrelated textures) as well as the own blob.
# Sufficiency score = shift caused by style-map shuffling, averaged over blob
# conditions; blob-channel strength = shift caused by swapping the blob alone.
# ============================================================
P23_CATS = ["chequered", "grid", "striped", "swirly", "flecked", "gauzy", "woven"]
P23_K = 3
donor_cats = ["bumpy", "blotchy"]          # unrelated blob donors
donor_blobs = [category_data[c]["blob_a_gen"] for c in donor_cats if c in category_data]

def decode_sm_with_blob(sm_largest, blob):
    smaps = {}
    for size in set(FMS):
        smaps[size] = sm_largest if size == largest else F.interpolate(
            sm_largest, size=size, mode="bilinear", align_corners=False)
    return decode_with_style_maps(blob, smaps)

print(f"{'category':12s}{'suff (deconf)':>14s}{'suff (own-blob)':>16s}{'blob channel':>13s}{'n':>3s}")
p23 = {}
for cat in P23_CATS:
    suff_d, suff_o, blobch = [], [], []
    for name, bf, si in load_category_exemplars(cat, k=P23_K, seed=31):
        try:
            img = load_texture(os.path.join(dtd_root, cat, name))
            own_blob, bf2, si2 = get_generator_inputs(img)
        except Exception:
            continue
        sm = build_style_maps(bf2, si2, FMS)
        shuf = block_shuffle_map(sm[largest], 1)          # b=1: full arrangement destruction
        blobs = [own_blob] + donor_blobs
        with torch.no_grad():
            # shuffle effect under each blob condition
            per_blob = []
            for bl in blobs:
                o = decode_sm_with_blob(sm[largest], bl)
                s = decode_sm_with_blob(shuf, bl)
                per_blob.append(vgg_dist.distance(s, o) / inter_scale)
            # blob-channel strength: same (unshuffled) style map, own vs donor blob
            o_own = decode_sm_with_blob(sm[largest], own_blob)
            bc = np.mean([vgg_dist.distance(
                decode_sm_with_blob(sm[largest], bl), o_own) / inter_scale
                for bl in donor_blobs])
        suff_o.append(per_blob[0])                        # own-blob (the old P16 number)
        suff_d.append(np.mean(per_blob[1:]))              # foreign blobs: channel neutralized
        blobch.append(bc)
    if suff_d:
        p23[cat] = (np.mean(suff_d), np.std(suff_d), np.mean(suff_o), np.mean(blobch))
        print(f"{cat:12s}{p23[cat][0]:9.2f}±{p23[cat][1]:4.2f}{p23[cat][2]:16.2f}"
              f"{p23[cat][3]:13.2f}{len(suff_d):3d}")

print("\nreading, per category:")
print("- suff(deconf) ~ suff(own-blob): the blob wasn't carrying the arrangement;")
print("  the old P16/P12 shuffle scores stand as marginal-sufficiency evidence.")
print("- suff(deconf) >> suff(own-blob), esp. for periodic textures: the blob WAS")
print("  reconstructing arrangement; chequered's b=1 survival was the H2' channel,")
print("  and the doc's 'generator resynthesizes periodicity' claim must go.")
print("- blob channel column large everywhere: H2' is strong regardless; report it")
print("  as a measured property of CNT either way.")

In [ ]:
# ============================================================
# Cell P24: Prop-2 core on a STRATIFIED random pair sample,
# with the corrected general t* formula.
# t* = (a0^2 - rho*a0*a1) / (a0^2 + a1^2 - 2*rho*a0*a1)   [clipped to [0,1]]
# mid/chord (exact, unequal amplitudes) = sqrt(a0^2+a1^2+2 rho a0 a1)/(a0+a1)
# ============================================================
random.seed(777)
_poolB = [p for p in all_pairs if p[0] in category_data and p[1] in category_data]
random.shuffle(_poolB)
P24_PAIRS = _poolB[:30]
n_within = sum(1 for p in P24_PAIRS if p[2] == "within")
print(f"stratified sample: {n_within} within-type, {len(P24_PAIRS)-n_within} cross-type\n")

def tstar_general(a0, a1, rho):
    denom = a0**2 + a1**2 - 2 * rho * a0 * a1
    if denom <= 1e-12: return 0.5
    return float(np.clip((a0**2 - rho * a0 * a1) / denom, 0, 1))

print(f"{'pair':32s}{'rho_res':>8s}{'mid_obs':>8s}{'mid_pred':>9s}{'t*_pred':>8s}{'t*_obs':>7s}{'exact':>7s}")
p24 = []
for ca, cb, ptype, _ in P24_PAIRS:
    sm_a = build_style_maps(category_data[ca]["bf_a"], category_data[ca]["si_a"], FMS)[largest][0]
    sm_b = build_style_maps(category_data[cb]["bf_a"], category_data[cb]["si_a"], FMS)[largest][0]
    Xf = sm_a.permute(1, 2, 0).reshape(-1, sm_a.shape[0])
    Yf = sm_b.permute(1, 2, 0).reshape(-1, sm_b.shape[0])
    idx = torch.randperm(Xf.shape[0], device=Xf.device)[:4096]
    X, Y = Xf[idx], Yf[idx]
    Xr, Yr = X - cm_flat[idx], Y - cm_flat[idx]           # scaffold-removed residuals
    a0, a1 = rms_amp(Xr), rms_amp(Yr)
    rho = rms_rho(Xr, Yr)
    # exact predictions, general amplitudes
    mid_pred = float(np.sqrt(a0**2 + a1**2 + 2 * rho * a0 * a1) / (a0 + a1))
    ts_pred = tstar_general(a0, a1, rho)
    # observations on a fine grid
    ts = np.linspace(0, 1, 41)
    amps = [rms_amp((1 - t) * Xr + t * Yr) for t in ts]
    chord = [(1 - t) * a0 + t * a1 for t in ts]
    mid_obs = amps[20] / chord[20]
    ts_obs = float(ts[int(np.argmin(amps))])
    # geodesic side
    Xe, Ye = X[:1500], Y[:1500]
    T = exact_ot_targets(Xe, Ye)
    ex = rms_amp(0.5 * Xe + 0.5 * T) / ((rms_amp(Xe) + rms_amp(T)) / 2)
    p24.append((f"{ca}_vs_{cb}", ptype, rho, mid_obs, mid_pred, ts_pred, ts_obs, ex))
    print(f"{ca+'_vs_'+cb:32s}{rho:8.3f}{mid_obs:8.3f}{mid_pred:9.3f}{ts_pred:8.2f}{ts_obs:7.2f}{ex:7.3f}")

arr = lambda i: np.array([r[i] for r in p24])
err_mid = np.abs(arr(3) - arr(4)); err_ts = np.abs(arr(5) - arr(6)); ex_ = arr(7)
wmask = np.array([r[1] == "within" for r in p24])
print(f"\nmid |obs-pred|:  all {err_mid.mean():.4f}   within {err_mid[wmask].mean():.4f}   cross {err_mid[~wmask].mean():.4f}")
print(f"t*  |obs-pred|:  all {err_ts.mean():.3f}    (grid resolution 0.025)")
print(f"exact mid/chord: all {ex_.mean():.3f} ± {ex_.std():.3f}   within {ex_[wmask].mean():.3f}   cross {ex_[~wmask].mean():.3f}")
print("\nverdict: within ~ cross on all three => the head-of-list bias didn't matter")
print("and coverage caveats can be replaced by this stratified result; a within/cross")
print("gap => earlier numbers were type-conditional and the doc must say on which types.")

In [ ]:
for cat in ["veined", "marbled", "cobwebbed"]:
    X, _ = (lambda sm: (sm.permute(1,2,0).reshape(-1, sm.shape[0])[torch.randperm(sm.shape[1]*sm.shape[2])[:1500]], None))(
        build_style_maps(category_data[cat]["bf_a"], category_data[cat]["si_a"], FMS)[largest][0])
    ref = build_style_maps(category_data["grooved"]["bf_a"], category_data["grooved"]["si_a"], FMS)[largest][0]
    R = ref.permute(1,2,0).reshape(-1, ref.shape[0])[torch.randperm(ref.shape[1]*ref.shape[2])[:1500]]
    T = exact_ot_targets(X, R)
    Xc, Tc = X - X.mean(0), T - T.mean(0)
    rho = (Xc*Tc).sum(1).mean().item() / (rms_amp(X) * rms_amp(T))
    print(f"{cat:12s} rho* vs grooved: {rho:.3f}")